<a href="https://colab.research.google.com/github/Jsolarte282000/app-recepcion/blob/main/auditoria_costos_semana_33.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BLOQUE 1

In [ ]:
# =========================================================
# BLOQUE 1 — IMPORTACIONES Y CARGA SEGURA DE LAS DOS BASES
# =========================================================

from google.colab import files
from pathlib import Path
from datetime import date
import pandas as pd
import numpy as np
import re
import unicodedata
import os

print("Selecciona exactamente estos dos archivos:")
print("1. Base de consumos")
print("2. Base de hectáreas")
print()

archivos_subidos = files.upload()

# ---------------------------------------------------------
# IDENTIFICAR ARCHIVOS EXCEL
# ---------------------------------------------------------

archivos_excel = [
    nombre
    for nombre in archivos_subidos.keys()
    if Path(nombre).suffix.lower() in {".xlsx", ".xls"}
]

if len(archivos_excel) != 2:
    raise ValueError(
        f"Debes subir exactamente 2 archivos Excel. "
        f"Se encontraron {len(archivos_excel)}: {archivos_excel}"
    )

# ---------------------------------------------------------
# INFORMACIÓN INICIAL
# ---------------------------------------------------------

hoy = date.today()
semana_calendario = int(hoy.isocalendar().week)
anio_calendario = int(hoy.isocalendar().year)

print("\n✅ Archivos cargados correctamente:\n")

for numero, archivo in enumerate(archivos_excel, start=1):
    tamano_mb = os.path.getsize(archivo) / (1024 ** 2)

    print(
        f"{numero}. {archivo}\n"
        f"   Tamaño: {tamano_mb:,.2f} MB"
    )

print("\nInformación del calendario:")
print(f"Fecha actual: {hoy}")
print(f"Año ISO actual: {anio_calendario}")
print(f"Semana ISO actual: {semana_calendario}")

print("\n✅ BLOQUE 1 TERMINADO")
print("Todavía no se ha modificado ni leído ninguna base.")

Selecciona exactamente estos dos archivos:
1. Base de consumos
2. Base de hectáreas



Saving Hectarias db.xlsx to Hectarias db (2).xlsx
Saving consumos 34.xlsx to consumos 34 (2).xlsx

✅ Archivos cargados correctamente:

1. Hectarias db (2).xlsx
   Tamaño: 0.11 MB
2. consumos 34 (2).xlsx
   Tamaño: 25.12 MB

Información del calendario:
Fecha actual: 2026-08-25
Año ISO actual: 2026
Semana ISO actual: 35

✅ BLOQUE 1 TERMINADO
Todavía no se ha modificado ni leído ninguna base.


# BLOQUE 2

In [ ]:
# =========================================================
# BLOQUE 2 — DETECCIÓN DE ARCHIVOS, HOJAS Y HEADERS
# =========================================================

def normalizar_texto_deteccion(valor):
    """
    Normaliza únicamente para identificar columnas.
    NO modifica los encabezados ni los datos originales.
    """
    if pd.isna(valor):
        return ""

    texto = str(valor).strip().upper()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )
    texto = re.sub(r"\s+", " ", texto)

    return texto


def detectar_fila_header(vista_previa, max_filas=20):
    """
    Busca la fila que más probablemente contiene los encabezados.
    """
    palabras_clave = {
        "FECHA",
        "SEMANA",
        "ANO",
        "ANIO",
        "BODEGA",
        "FINCA",
        "FAMILIA",
        "AREA",
        "TOTAL",
        "HECTARIAS",
        "HECTAREAS",
        "GRUPOS DE MATERIALES",
        "GRUPO DE MATERIALES",
        "NOMBRE",
        "USUARIO",
        "VARIEDAD",
        "PRODUCTO"
    }

    mejor_fila = None
    mejor_puntaje = -1

    limite = min(max_filas, len(vista_previa))

    for numero_fila in range(limite):

        valores_fila = {
            normalizar_texto_deteccion(valor)
            for valor in vista_previa.iloc[numero_fila].tolist()
            if normalizar_texto_deteccion(valor) != ""
        }

        puntaje = len(valores_fila.intersection(palabras_clave))

        if puntaje > mejor_puntaje:
            mejor_puntaje = puntaje
            mejor_fila = numero_fila

    return mejor_fila, mejor_puntaje


def puntuar_tipo_base(headers):
    """
    Calcula qué tan probable es que una hoja sea:
    - base de consumos;
    - base de hectáreas.
    """
    columnas = {
        normalizar_texto_deteccion(columna)
        for columna in headers
        if normalizar_texto_deteccion(columna) != ""
    }

    puntaje_consumos = 0
    puntaje_hectareas = 0

    # -----------------------------
    # Puntaje base de consumos
    # -----------------------------
    if "TOTAL" in columnas:
        puntaje_consumos += 4

    if "FECHA" in columnas:
        puntaje_consumos += 2

    if "SEMANA" in columnas:
        puntaje_consumos += 2

    if "BODEGA" in columnas or "FINCA" in columnas:
        puntaje_consumos += 2

    if (
        "FAMILIA" in columnas
        or "GRUPOS DE MATERIALES" in columnas
        or "GRUPO DE MATERIALES" in columnas
    ):
        puntaje_consumos += 2

    if "AREA" in columnas:
        puntaje_consumos += 1

    if "NOMBRE" in columnas or "USUARIO" in columnas:
        puntaje_consumos += 1

    # -----------------------------
    # Puntaje base de hectáreas
    # -----------------------------
    if "HECTARIAS" in columnas or "HECTAREAS" in columnas:
        puntaje_hectareas += 6

    if "SEMANA" in columnas:
        puntaje_hectareas += 2

    if "FINCA" in columnas or "BODEGA" in columnas:
        puntaje_hectareas += 2

    if "FAMILIA" in columnas:
        puntaje_hectareas += 2

    if (
        "ANO" in columnas
        or "ANIO" in columnas
        or "AÑO" in columnas
    ):
        puntaje_hectareas += 1

    return puntaje_consumos, puntaje_hectareas


# =========================================================
# 1. INSPECCIONAR TODOS LOS ARCHIVOS Y TODAS LAS HOJAS
# =========================================================

candidatos = []

print("Inspeccionando archivos y hojas...\n")

for archivo in archivos_excel:

    libro = pd.ExcelFile(archivo)

    print(f"Archivo: {archivo}")
    print(f"Hojas encontradas: {libro.sheet_names}")

    for hoja in libro.sheet_names:

        try:
            vista_previa = pd.read_excel(
                archivo,
                sheet_name=hoja,
                header=None,
                nrows=25,
                dtype=object
            )

            if vista_previa.empty:
                print(f"  - Hoja '{hoja}': vacía")
                continue

            fila_header, puntaje_header = detectar_fila_header(
                vista_previa
            )

            headers_fuente = vista_previa.iloc[
                fila_header
            ].tolist()

            puntaje_consumos, puntaje_hectareas = (
                puntuar_tipo_base(headers_fuente)
            )

            candidatos.append({
                "archivo": archivo,
                "hoja": hoja,
                "fila_header": fila_header,
                "puntaje_header": puntaje_header,
                "puntaje_consumos": puntaje_consumos,
                "puntaje_hectareas": puntaje_hectareas,
                "headers_fuente": headers_fuente
            })

            print(
                f"  - Hoja '{hoja}' | "
                f"header probable: fila {fila_header + 1} | "
                f"consumos: {puntaje_consumos} | "
                f"hectáreas: {puntaje_hectareas}"
            )

        except Exception as error:
            print(
                f"  - No se pudo inspeccionar la hoja "
                f"'{hoja}': {error}"
            )

    print()


if not candidatos:
    raise ValueError(
        "No fue posible inspeccionar ninguna hoja de los archivos."
    )


# =========================================================
# 2. SELECCIONAR LOS MEJORES CANDIDATOS
# =========================================================

candidato_consumos = max(
    candidatos,
    key=lambda x: x["puntaje_consumos"]
)

candidato_hectareas = max(
    candidatos,
    key=lambda x: x["puntaje_hectareas"]
)

if candidato_consumos["puntaje_consumos"] < 6:
    raise ValueError(
        "No se identificó con suficiente seguridad "
        "la base de consumos."
    )

if candidato_hectareas["puntaje_hectareas"] < 7:
    raise ValueError(
        "No se identificó con suficiente seguridad "
        "la base de hectáreas."
    )

if (
    candidato_consumos["archivo"] == candidato_hectareas["archivo"]
    and candidato_consumos["hoja"] == candidato_hectareas["hoja"]
):
    raise ValueError(
        "La misma hoja fue identificada como consumos y hectáreas. "
        "Debemos revisarla antes de continuar."
    )


archivo_consumos = candidato_consumos["archivo"]
hoja_consumos = candidato_consumos["hoja"]
fila_header_consumos = candidato_consumos["fila_header"]

archivo_hectareas = candidato_hectareas["archivo"]
hoja_hectareas = candidato_hectareas["hoja"]
fila_header_hectareas = candidato_hectareas["fila_header"]


# =========================================================
# 3. CARGAR LAS DOS BASES CONSERVANDO SUS HEADERS
# =========================================================

consumos_original = pd.read_excel(
    archivo_consumos,
    sheet_name=hoja_consumos,
    header=fila_header_consumos,
    dtype=object
)

hectareas_original = pd.read_excel(
    archivo_hectareas,
    sheet_name=hoja_hectareas,
    header=fila_header_hectareas,
    dtype=object
)

# Copias independientes para trabajar después
consumos_trabajo = consumos_original.copy(deep=True)
hectareas_trabajo = hectareas_original.copy(deep=True)

# Guardar los encabezados antes de cualquier modificación
headers_consumos_originales = list(consumos_original.columns)
headers_hectareas_originales = list(hectareas_original.columns)


# =========================================================
# 4. AUDITAR HEADERS
# =========================================================

def auditar_headers(nombre_base, dataframe, headers_fuente):

    print(f"\n{'=' * 70}")
    print(f"AUDITORÍA DE HEADERS: {nombre_base}")
    print(f"{'=' * 70}")

    print(f"Cantidad de columnas: {len(dataframe.columns)}")
    print()

    for posicion, columna in enumerate(dataframe.columns, start=1):
        print(f"{posicion:>3}. {repr(columna)}")

    columnas_pandas_duplicadas = (
        dataframe.columns[
            dataframe.columns.duplicated()
        ].tolist()
    )

    headers_no_vacios = [
        str(valor).strip()
        for valor in headers_fuente
        if not pd.isna(valor) and str(valor).strip() != ""
    ]

    headers_fuente_duplicados = sorted({
        header
        for header in headers_no_vacios
        if headers_no_vacios.count(header) > 1
    })

    columnas_sin_nombre = [
        columna
        for columna in dataframe.columns
        if str(columna).startswith("Unnamed:")
    ]

    print("\nResultado de auditoría:")

    if columnas_pandas_duplicadas:
        print(
            "⚠️ Pandas conserva columnas duplicadas:",
            columnas_pandas_duplicadas
        )
    else:
        print("✅ No hay nombres duplicados dentro del DataFrame.")

    if headers_fuente_duplicados:
        print(
            "⚠️ El Excel tiene headers repetidos en la fila original:",
            headers_fuente_duplicados
        )
    else:
        print("✅ El Excel no presenta headers repetidos visibles.")

    if columnas_sin_nombre:
        print(
            "⚠️ Columnas sin encabezado:",
            columnas_sin_nombre
        )
    else:
        print("✅ Todas las columnas tienen encabezado.")


# =========================================================
# 5. MOSTRAR RESULTADOS
# =========================================================

print("\n✅ BASES IDENTIFICADAS\n")

print("BASE DE CONSUMOS")
print(f"Archivo: {archivo_consumos}")
print(f"Hoja: {hoja_consumos}")
print(f"Fila de headers: {fila_header_consumos + 1}")
print(f"Filas cargadas: {len(consumos_original):,}")
print(f"Columnas cargadas: {len(consumos_original.columns)}")

print("\nBASE DE HECTÁREAS")
print(f"Archivo: {archivo_hectareas}")
print(f"Hoja: {hoja_hectareas}")
print(f"Fila de headers: {fila_header_hectareas + 1}")
print(f"Filas cargadas: {len(hectareas_original):,}")
print(f"Columnas cargadas: {len(hectareas_original.columns)}")

auditar_headers(
    "CONSUMOS",
    consumos_original,
    candidato_consumos["headers_fuente"]
)

auditar_headers(
    "HECTÁREAS",
    hectareas_original,
    candidato_hectareas["headers_fuente"]
)

print("\nPRIMERAS 3 FILAS DE CONSUMOS:")
display(consumos_original.head(3))

print("\nPRIMERAS 3 FILAS DE HECTÁREAS:")
display(hectareas_original.head(3))

print("\n✅ BLOQUE 2 TERMINADO")
print("Las bases fueron leídas, pero todavía no se modificó ningún dato.")

Inspeccionando archivos y hojas...

Archivo: Hectarias db (2).xlsx
Hojas encontradas: ['Hoja1']
  - Hoja 'Hoja1' | header probable: fila 1 | consumos: 6 | hectáreas: 13

Archivo: consumos 34 (2).xlsx
Hojas encontradas: ['Hoja2']
  - Hoja 'Hoja2' | header probable: fila 1 | consumos: 14 | hectáreas: 4


✅ BASES IDENTIFICADAS

BASE DE CONSUMOS
Archivo: consumos 34 (2).xlsx
Hoja: Hoja2
Fila de headers: 1
Filas cargadas: 376,489
Columnas cargadas: 16

BASE DE HECTÁREAS
Archivo: Hectarias db (2).xlsx
Hoja: Hoja1
Fila de headers: 1
Filas cargadas: 3,718
Columnas cargadas: 5

AUDITORÍA DE HEADERS: CONSUMOS
Cantidad de columnas: 16

  1. 'BODEGA'
  2. 'FECHA'
  3. 'SEMANA'
  4. 'MES'
  5. 'DOC'
  6. 'CODIGO'
  7. 'GRUPO'
  8. 'GRUPOS DE MATERIALES'
  9. 'NOMBRE'
 10. 'UND'
 11. 'CANTIDAD'
 12. 'PROM'
 13. 'TOTAL'
 14. 'CENTRO DE COSTO'
 15. 'USUARIO'
 16. 'AREA'

Resultado de auditoría:
✅ No hay nombres duplicados dentro del DataFrame.
✅ El Excel no presenta headers repetidos visibles.
✅ Todas

,BODEGA,FECHA,SEMANA,MES,DOC,CODIGO,GRUPO,GRUPOS DE MATERIALES,NOMBRE,UND,CANTIDAD,PROM,TOTAL,CENTRO DE COSTO,USUARIO,AREA
0,Principal,46029,2,enero,10134260,103020005,10302,ACEITES Y COMBUSTIBLES,DIESEL,Gl,10,2.41,24.12,Gypsophila xlece CIF Postcosecha,Mónica Yungan,POSTCOSECHA
1,Principal,46030,2,enero,10134317,103020005,10302,ACEITES Y COMBUSTIBLES,DIESEL,Gl,20.9,2.41,50.41,Gypsophila xlece CIF Postcosecha,Mónica Yungan,POSTCOSECHA
2,Principal,46030,2,enero,10134318,103020005,10302,ACEITES Y COMBUSTIBLES,DIESEL,Gl,5.8,2.41,13.99,Gypsophila xlece CIF Postcosecha,Mónica Yungan,POSTCOSECHA



PRIMERAS 3 FILAS DE HECTÁREAS:


,AÑO,SEMANA,HECTARIAS,FINCA,FAMILIA
0,2024,1,3.040362,MALCHINGUI,ASTRANTIAS
1,2024,2,3.097251,MALCHINGUI,ASTRANTIAS
2,2024,3,3.18517,MALCHINGUI,ASTRANTIAS



✅ BLOQUE 2 TERMINADO
Las bases fueron leídas, pero todavía no se modificó ningún dato.


# BLOQUE 3

In [ ]:
# =========================================================
# BLOQUE 3 — DETECTAR Y VALIDAR LA SEMANA CAÍDA
# =========================================================

# IMPORTANTE:
# - No modifica las bases originales.
# - No cruza todavía consumos con hectáreas.
# - Detecta la última semana cerrada disponible.
# - Valida SEMANA contra FECHA.
# - Revisa disponibilidad y duplicados en hectáreas.

# =========================================================
# 1. PROTEGER HEADERS ORIGINALES
# =========================================================

if list(consumos_original.columns) != headers_consumos_originales:
    raise ValueError(
        "Los headers de consumos fueron modificados."
    )

if list(hectareas_original.columns) != headers_hectareas_originales:
    raise ValueError(
        "Los headers de hectáreas fueron modificados."
    )

consumos_periodo = consumos_original.copy(deep=True)
hectareas_periodo = hectareas_original.copy(deep=True)

print("✅ Los headers originales permanecen intactos.")


# =========================================================
# 2. CALENDARIO ACTUAL
# =========================================================

fecha_actual = (
    pd.Timestamp.now(tz="America/Guayaquil")
    .tz_localize(None)
    .normalize()
)

iso_actual = fecha_actual.isocalendar()

ANIO_ACTUAL = int(iso_actual.year)
SEMANA_ACTUAL = int(iso_actual.week)
PERIODO_ACTUAL = ANIO_ACTUAL * 100 + SEMANA_ACTUAL

# Semana anterior según calendario
fecha_semana_anterior = fecha_actual - pd.Timedelta(days=7)
iso_semana_anterior = fecha_semana_anterior.isocalendar()

ANIO_CAIDO_ESPERADO = int(iso_semana_anterior.year)
SEMANA_CAIDA_ESPERADA = int(iso_semana_anterior.week)

PERIODO_CAIDO_ESPERADO = (
    ANIO_CAIDO_ESPERADO * 100
    + SEMANA_CAIDA_ESPERADA
)

print("\n" + "=" * 75)
print("CALENDARIO DEL PROCESO")
print("=" * 75)

print(f"Fecha actual: {fecha_actual.date()}")
print(f"Semana actual abierta: {ANIO_ACTUAL} - semana {SEMANA_ACTUAL}")
print(
    f"Semana caída esperada por calendario: "
    f"{ANIO_CAIDO_ESPERADO} - semana {SEMANA_CAIDA_ESPERADA}"
)


# =========================================================
# 3. PREPARAR FECHA DE CONSUMOS
# =========================================================

fecha_numerica_excel = pd.to_numeric(
    consumos_periodo["FECHA"],
    errors="coerce"
)

consumos_periodo["_FECHA_DT"] = pd.Series(
    pd.NaT,
    index=consumos_periodo.index,
    dtype="datetime64[ns]"
)

mask_fecha_excel = fecha_numerica_excel.notna()

consumos_periodo.loc[
    mask_fecha_excel,
    "_FECHA_DT"
] = pd.to_datetime(
    fecha_numerica_excel.loc[mask_fecha_excel],
    unit="D",
    origin="1899-12-30",
    errors="coerce"
)

consumos_periodo.loc[
    ~mask_fecha_excel,
    "_FECHA_DT"
] = pd.to_datetime(
    consumos_periodo.loc[
        ~mask_fecha_excel,
        "FECHA"
    ],
    errors="coerce"
)

mask_fecha_invalida = consumos_periodo["_FECHA_DT"].isna()
cantidad_fechas_invalidas = int(mask_fecha_invalida.sum())

print("\n" + "=" * 75)
print("VALIDACIÓN DE FECHAS")
print("=" * 75)

print(f"Registros revisados: {len(consumos_periodo):,}")
print(f"Fechas inválidas: {cantidad_fechas_invalidas:,}")

if cantidad_fechas_invalidas > 0:

    display(
        consumos_periodo.loc[
            mask_fecha_invalida,
            [
                "FECHA",
                "SEMANA",
                "BODEGA",
                "TOTAL"
            ]
        ].head(30)
    )

    raise ValueError(
        "Existen fechas inválidas. "
        "No podemos determinar correctamente la semana."
    )

print(
    f"Fecha mínima de consumos: "
    f"{consumos_periodo['_FECHA_DT'].min().date()}"
)

print(
    f"Fecha máxima de consumos: "
    f"{consumos_periodo['_FECHA_DT'].max().date()}"
)


# =========================================================
# 4. CALCULAR AÑO Y SEMANA ISO DESDE FECHA
# =========================================================

calendario_consumos = (
    consumos_periodo["_FECHA_DT"]
    .dt
    .isocalendar()
)

consumos_periodo["_AÑO_FECHA"] = (
    calendario_consumos["year"]
    .astype("Int64")
)

consumos_periodo["_SEMANA_FECHA"] = (
    calendario_consumos["week"]
    .astype("Int64")
)

consumos_periodo["_PERIODO_FECHA"] = (
    consumos_periodo["_AÑO_FECHA"] * 100
    + consumos_periodo["_SEMANA_FECHA"]
).astype("Int64")


# =========================================================
# 5. VALIDAR LA COLUMNA SEMANA ORIGINAL
# =========================================================

semana_texto = (
    consumos_periodo["SEMANA"]
    .astype("string")
    .str.strip()
    .str.upper()
)

semana_extraida = semana_texto.str.extract(
    r"^\s*(\d{1,2})",
    expand=False
)

consumos_periodo["_SEMANA_ORIGINAL"] = pd.to_numeric(
    semana_extraida,
    errors="coerce"
).astype("Int64")

mask_semana_invalida = (
    consumos_periodo["_SEMANA_ORIGINAL"].isna()
    | ~consumos_periodo["_SEMANA_ORIGINAL"].between(1, 53)
)

cantidad_semanas_invalidas = int(mask_semana_invalida.sum())

mask_semana_no_coincide = (
    consumos_periodo["_SEMANA_ORIGINAL"]
    .fillna(-1)
    .astype(int)
    !=
    consumos_periodo["_SEMANA_FECHA"]
    .fillna(-2)
    .astype(int)
)

cantidad_no_coincide = int(mask_semana_no_coincide.sum())

print("\n" + "=" * 75)
print("VALIDACIÓN DE SEMANA ORIGINAL")
print("=" * 75)

print(
    f"Semanas inválidas o fuera de 1–53: "
    f"{cantidad_semanas_invalidas:,}"
)

print(
    f"Registros donde SEMANA no coincide con FECHA: "
    f"{cantidad_no_coincide:,}"
)

if cantidad_semanas_invalidas > 0:

    print("\n⚠️ Ejemplos de semanas inválidas:")

    display(
        consumos_periodo.loc[
            mask_semana_invalida,
            [
                "FECHA",
                "SEMANA",
                "_SEMANA_FECHA",
                "MES",
                "BODEGA"
            ]
        ].head(30)
    )

    raise ValueError(
        "Existen semanas inválidas en la base de consumos."
    )

if cantidad_no_coincide > 0:

    print("\n⚠️ Ejemplos donde SEMANA no coincide con FECHA:")

    display(
        consumos_periodo.loc[
            mask_semana_no_coincide,
            [
                "FECHA",
                "SEMANA",
                "_SEMANA_FECHA",
                "MES",
                "BODEGA",
                "TOTAL"
            ]
        ].head(30)
    )

    raise ValueError(
        "La columna SEMANA no coincide con la semana "
        "calculada desde FECHA."
    )

print(
    "✅ Todas las semanas coinciden con la fecha correspondiente."
)


# =========================================================
# 6. CONVERTIR TOTAL A NUMÉRICO
# =========================================================

consumos_periodo["_TOTAL_NUM"] = pd.to_numeric(
    consumos_periodo["TOTAL"],
    errors="coerce"
)

# Algunos registros pueden venir como texto con coma decimal
# (ej.: "2,82"). Solo se corrigen los que no pudieron
# convertirse directamente; la columna TOTAL original no se modifica.
mask_total_texto = (
    consumos_periodo["TOTAL"].notna()
    & consumos_periodo["_TOTAL_NUM"].isna()
)

if mask_total_texto.any():
    total_texto_normalizado = (
        consumos_periodo.loc[
            mask_total_texto,
            "TOTAL"
        ]
        .astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False)
    )

    consumos_periodo.loc[
        mask_total_texto,
        "_TOTAL_NUM"
    ] = pd.to_numeric(
        total_texto_normalizado,
        errors="coerce"
    )

mask_total_invalido = (
    consumos_periodo["TOTAL"].notna()
    & consumos_periodo["_TOTAL_NUM"].isna()
)

cantidad_totales_invalidos = int(mask_total_invalido.sum())

if cantidad_totales_invalidos > 0:

    print(
        f"\n⚠️ Totales no numéricos encontrados: "
        f"{cantidad_totales_invalidos:,}"
    )

    display(
        consumos_periodo.loc[
            mask_total_invalido,
            [
                "FECHA",
                "SEMANA",
                "BODEGA",
                "TOTAL"
            ]
        ].head(30)
    )

    raise ValueError(
        "Existen valores no numéricos en TOTAL."
    )


# =========================================================
# 7. MOSTRAR LAS ÚLTIMAS SEMANAS DE LA BASE
# =========================================================

resumen_semanas = (
    consumos_periodo
    .groupby(
        [
            "_AÑO_FECHA",
            "_SEMANA_FECHA",
            "_PERIODO_FECHA"
        ],
        dropna=False
    )
    .agg(
        REGISTROS=("_TOTAL_NUM", "size"),
        TOTAL=("_TOTAL_NUM", "sum"),
        FECHA_MIN=("_FECHA_DT", "min"),
        FECHA_MAX=("_FECHA_DT", "max")
    )
    .reset_index()
    .sort_values("_PERIODO_FECHA")
)

print("\n" + "=" * 75)
print("ÚLTIMAS SEMANAS DISPONIBLES EN CONSUMOS")
print("=" * 75)

display(
    resumen_semanas.tail(10)[
        [
            "_AÑO_FECHA",
            "_SEMANA_FECHA",
            "FECHA_MIN",
            "FECHA_MAX",
            "REGISTROS",
            "TOTAL"
        ]
    ]
)


# =========================================================
# 8. EXCLUIR LA SEMANA ACTUAL POR ESTAR ABIERTA
# =========================================================

mask_periodo_cerrado = (
    consumos_periodo["_PERIODO_FECHA"]
    < PERIODO_ACTUAL
)

consumos_cerrados = (
    consumos_periodo.loc[
        mask_periodo_cerrado
    ]
    .copy()
)

if consumos_cerrados.empty:
    raise ValueError(
        "No existen semanas cerradas anteriores "
        "a la semana actual."
    )


# =========================================================
# 9. DETECTAR ÚLTIMA SEMANA CERRADA DISPONIBLE
# =========================================================

PERIODO_PROCESO = int(
    consumos_cerrados["_PERIODO_FECHA"].max()
)

ANIO_PROCESO = int(PERIODO_PROCESO // 100)
SEMANA_PROCESO = int(PERIODO_PROCESO % 100)

mask_semana_proceso = (
    consumos_periodo["_PERIODO_FECHA"]
    == PERIODO_PROCESO
)

consumos_semana_proceso = (
    consumos_periodo.loc[
        mask_semana_proceso
    ]
    .copy()
)

REGISTROS_SEMANA_PROCESO = len(
    consumos_semana_proceso
)

TOTAL_SEMANA_PROCESO = float(
    consumos_semana_proceso["_TOTAL_NUM"].sum()
)

FECHA_MIN_PROCESO = (
    consumos_semana_proceso["_FECHA_DT"].min()
)

FECHA_MAX_PROCESO = (
    consumos_semana_proceso["_FECHA_DT"].max()
)

print("\n" + "=" * 75)
print("SEMANA CAÍDA SELECCIONADA")
print("=" * 75)

print(f"ANIO_PROCESO: {ANIO_PROCESO}")
print(f"SEMANA_PROCESO: {SEMANA_PROCESO}")
print(f"PERIODO_PROCESO: {PERIODO_PROCESO}")

print(
    f"Rango de fechas: "
    f"{FECHA_MIN_PROCESO.date()} a "
    f"{FECHA_MAX_PROCESO.date()}"
)

print(
    f"Registros de la semana {SEMANA_PROCESO}: "
    f"{REGISTROS_SEMANA_PROCESO:,}"
)

print(
    f"Total original de la semana {SEMANA_PROCESO}: "
    f"${TOTAL_SEMANA_PROCESO:,.2f}"
)


# =========================================================
# 10. COMPARAR CON LA SEMANA CAÍDA DEL CALENDARIO
# =========================================================

if PERIODO_PROCESO == PERIODO_CAIDO_ESPERADO:

    print(
        "\n✅ La base contiene exactamente la semana "
        "caída esperada por calendario."
    )

elif PERIODO_PROCESO < PERIODO_CAIDO_ESPERADO:

    diferencia_periodos = (
        PERIODO_CAIDO_ESPERADO
        - PERIODO_PROCESO
    )

    print(
        "\n⚠️ La base no llega todavía a la semana "
        "caída esperada por calendario."
    )

    print(
        f"Se procesará correctamente la última semana "
        f"disponible: semana {SEMANA_PROCESO}."
    )

else:

    raise ValueError(
        "La última semana de la base es posterior "
        "a la semana caída esperada."
    )


# =========================================================
# 11. PREPARAR HECTÁREAS SIN MODIFICAR HEADERS
# =========================================================

hectareas_periodo["_AÑO_NUM"] = pd.to_numeric(
    hectareas_periodo["AÑO"],
    errors="coerce"
).astype("Int64")

hectareas_periodo["_SEMANA_NUM"] = pd.to_numeric(
    hectareas_periodo["SEMANA"],
    errors="coerce"
).astype("Int64")

hectareas_periodo["_HECTARIAS_NUM"] = pd.to_numeric(
    hectareas_periodo["HECTARIAS"],
    errors="coerce"
)


def normalizar_llave_temporal(serie):
    """
    Normaliza para auditar llaves.
    No modifica la columna original.
    """
    return (
        serie
        .astype("string")
        .str.strip()
        .str.upper()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.replace(r"\s+", " ", regex=True)
    )


hectareas_periodo["_FINCA_KEY"] = normalizar_llave_temporal(
    hectareas_periodo["FINCA"]
)

hectareas_periodo["_FAMILIA_KEY"] = normalizar_llave_temporal(
    hectareas_periodo["FAMILIA"]
)


# =========================================================
# 12. AUDITAR AÑOS INVÁLIDOS EN HECTÁREAS
# =========================================================

mask_anio_hectareas_invalido = (
    hectareas_periodo["_AÑO_NUM"].isna()
    | ~hectareas_periodo["_AÑO_NUM"].between(2000, 2100)
)

cantidad_anios_invalidos = int(
    mask_anio_hectareas_invalido.sum()
)

print("\n" + "=" * 75)
print("VALIDACIÓN DE AÑOS EN HECTÁREAS")
print("=" * 75)

print(
    "Años encontrados:",
    sorted(
        hectareas_periodo["_AÑO_NUM"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
)

print(
    f"Registros con año inválido: "
    f"{cantidad_anios_invalidos:,}"
)

if cantidad_anios_invalidos > 0:

    print(
        "\n⚠️ Se encontraron años sospechosos. "
        "No se modificarán automáticamente."
    )

    display(
        hectareas_periodo.loc[
            mask_anio_hectareas_invalido,
            [
                "AÑO",
                "SEMANA",
                "HECTARIAS",
                "FINCA",
                "FAMILIA"
            ]
        ].head(30)
    )


# =========================================================
# 13. EXTRAER HECTÁREAS DE LA SEMANA DE PROCESO
# =========================================================

mask_hectareas_semana_proceso = (
    (
        hectareas_periodo["_AÑO_NUM"]
        == ANIO_PROCESO
    )
    &
    (
        hectareas_periodo["_SEMANA_NUM"]
        == SEMANA_PROCESO
    )
)

hectareas_semana_proceso = (
    hectareas_periodo.loc[
        mask_hectareas_semana_proceso
    ]
    .copy()
)

print("\n" + "=" * 75)
print("HECTÁREAS PARA LA SEMANA DE PROCESO")
print("=" * 75)

print(
    f"Registros encontrados para "
    f"{ANIO_PROCESO}, semana {SEMANA_PROCESO}: "
    f"{len(hectareas_semana_proceso):,}"
)

if hectareas_semana_proceso.empty:

    raise ValueError(
        f"No existen hectáreas para "
        f"{ANIO_PROCESO}, semana {SEMANA_PROCESO}."
    )


# =========================================================
# 14. DETECTAR LLAVES DUPLICADAS EN HECTÁREAS
# =========================================================

llaves_hectareas = [
    "_AÑO_NUM",
    "_SEMANA_NUM",
    "_FINCA_KEY",
    "_FAMILIA_KEY"
]

mask_duplicados_hectareas = (
    hectareas_semana_proceso
    .duplicated(
        subset=llaves_hectareas,
        keep=False
    )
)

duplicados_hectareas_semana = (
    hectareas_semana_proceso.loc[
        mask_duplicados_hectareas,
        [
            "AÑO",
            "SEMANA",
            "FINCA",
            "FAMILIA",
            "HECTARIAS",
            "_AÑO_NUM",
            "_SEMANA_NUM",
            "_FINCA_KEY",
            "_FAMILIA_KEY"
        ]
    ]
    .sort_values(
        [
            "_FINCA_KEY",
            "_FAMILIA_KEY"
        ]
    )
)

print(
    f"Filas involucradas en llaves duplicadas: "
    f"{len(duplicados_hectareas_semana):,}"
)

if duplicados_hectareas_semana.empty:

    print(
        "✅ No hay llaves duplicadas en las hectáreas "
        "de la semana procesada."
    )

else:

    print(
        "\n⚠️ Estas llaves duplicadas deben corregirse "
        "antes del merge:"
    )

    display(
        duplicados_hectareas_semana[
            [
                "AÑO",
                "SEMANA",
                "FINCA",
                "FAMILIA",
                "HECTARIAS"
            ]
        ]
    )


# =========================================================
# 15. CONFIRMAR QUE NO CAMBIARON LOS HEADERS
# =========================================================

assert list(consumos_original.columns) == headers_consumos_originales
assert list(hectareas_original.columns) == headers_hectareas_originales

print("\n" + "=" * 75)
print("RESULTADO DEL BLOQUE 3")
print("=" * 75)

print(f"ANIO_PROCESO = {ANIO_PROCESO}")
print(f"SEMANA_PROCESO = {SEMANA_PROCESO}")
print(f"PERIODO_PROCESO = {PERIODO_PROCESO}")

print(
    f"REGISTROS_SEMANA_PROCESO = "
    f"{REGISTROS_SEMANA_PROCESO:,}"
)

print(
    f"TOTAL_SEMANA_PROCESO = "
    f"${TOTAL_SEMANA_PROCESO:,.2f}"
)

print(
    f"REGISTROS_HECTAREAS_PROCESO = "
    f"{len(hectareas_semana_proceso):,}"
)

print(
    f"FILAS_DUPLICADAS_HECTAREAS = "
    f"{len(duplicados_hectareas_semana):,}"
)

print("✅ Los headers originales permanecen intactos.")
print("✅ BLOQUE 3 TERMINADO")
print("Todavía no se ha realizado ningún merge.")



# =======================================

✅ Los headers originales permanecen intactos.

CALENDARIO DEL PROCESO
Fecha actual: 2026-08-25
Semana actual abierta: 2026 - semana 35
Semana caída esperada por calendario: 2026 - semana 34

VALIDACIÓN DE FECHAS
Registros revisados: 376,489
Fechas inválidas: 0
Fecha mínima de consumos: 2026-01-07
Fecha máxima de consumos: 2026-08-23

VALIDACIÓN DE SEMANA ORIGINAL
Semanas inválidas o fuera de 1–53: 0
Registros donde SEMANA no coincide con FECHA: 0
✅ Todas las semanas coinciden con la fecha correspondiente.

ÚLTIMAS SEMANAS DISPONIBLES EN CONSUMOS


,_AÑO_FECHA,_SEMANA_FECHA,FECHA_MIN,FECHA_MAX,REGISTROS,TOTAL
23,2026,25,2026-06-15,2026-06-21,12462,325923.659591
24,2026,26,2026-06-22,2026-06-28,9567,365388.211386
25,2026,27,2026-07-03,2026-07-05,8219,321785.269271
26,2026,28,2026-07-06,2026-07-12,12274,324896.209942
27,2026,29,2026-07-13,2026-07-19,9533,326561.200000
28,2026,30,2026-07-20,2026-07-25,12085,328160.030000
29,2026,31,2026-07-27,2026-08-02,11155,362927.859584
30,2026,32,2026-08-03,2026-08-09,10848,356018.736231
31,2026,33,2026-08-10,2026-08-16,12645,353968.280000
32,2026,34,2026-08-17,2026-08-23,12042,336487.660000



SEMANA CAÍDA SELECCIONADA
ANIO_PROCESO: 2026
SEMANA_PROCESO: 34
PERIODO_PROCESO: 202634
Rango de fechas: 2026-08-17 a 2026-08-23
Registros de la semana 34: 12,042
Total original de la semana 34: $336,487.66

✅ La base contiene exactamente la semana caída esperada por calendario.

VALIDACIÓN DE AÑOS EN HECTÁREAS
Años encontrados: [2024, 2025, 2026]
Registros con año inválido: 0

HECTÁREAS PARA LA SEMANA DE PROCESO
Registros encontrados para 2026, semana 34: 21
Filas involucradas en llaves duplicadas: 0
✅ No hay llaves duplicadas en las hectáreas de la semana procesada.

RESULTADO DEL BLOQUE 3
ANIO_PROCESO = 2026
SEMANA_PROCESO = 34
PERIODO_PROCESO = 202634
REGISTROS_SEMANA_PROCESO = 12,042
TOTAL_SEMANA_PROCESO = $336,487.66
REGISTROS_HECTAREAS_PROCESO = 21
FILAS_DUPLICADAS_HECTAREAS = 0
✅ Los headers originales permanecen intactos.
✅ BLOQUE 3 TERMINADO
Todavía no se ha realizado ningún merge.


# BLOQUE 4

In [ ]:
# BLOQUE 4 — HOMOLOGACIÓN DEFINITIVA DE FAMILIAS Y ÁREAS
# =========================================================
#
# CORRECCIONES:
#
# GYPSOPHILA:
# - GYPSOPHILA
# - GYPSOPHILIA
# - GYPSO
# - GYPSO SMALL BLOOM
# - SMALL BLOOM
# Todo queda como GYPSOPHILA.
#
# SOLIDAGO:
# - SOLIDAGO
# - SOLIDADO
# Todo queda como SOLIDAGO.
#
# ÁREA:
# - Se obtiene directamente desde CENTRO DE COSTO.
# - Completa áreas vacías.
# - Corrige áreas existentes que contradicen el centro.
# - Esto corrige los registros SOLIDADO que vienen como
#   ADMINISTRACION, aunque el centro indica PROPAGACION.
#
# DEVOLUCIONES:
# - Sin CENTRO DE COSTO y TOTAL <= 0:
#       FAMILIA_DESGLOSE = SIN CENTRO DE COSTO
#       FAMILIA = OTROS
#
# PROTECCIONES:
# - No elimina registros.
# - No multiplica registros.
# - No cambia valores de TOTAL.
# - No modifica consumos_original.
# - Todavía no realiza el cruce con hectáreas.
# =========================================================


# =========================================================
# 1. IMPORTACIONES Y VALIDACIÓN INICIAL
# =========================================================

import pandas as pd
import numpy as np
import re
import unicodedata


if list(consumos_original.columns) != headers_consumos_originales:

    raise ValueError(
        "Los headers originales de consumos fueron modificados."
    )


columnas_obligatorias = [
    "BODEGA",
    "FECHA",
    "SEMANA",
    "TOTAL",
    "CENTRO DE COSTO",
    "AREA",
    "GRUPOS DE MATERIALES",
    "NOMBRE",
    "USUARIO"
]


columnas_faltantes = [
    columna
    for columna in columnas_obligatorias
    if columna not in consumos_original.columns
]


if columnas_faltantes:

    raise ValueError(
        f"Faltan columnas obligatorias: "
        f"{columnas_faltantes}"
    )


# Siempre reconstruir df desde la base original.
df = consumos_original.copy(deep=True)


FILAS_ANTES_BLOQUE_4 = len(df)


total_auditoria_inicial = pd.to_numeric(
    df["TOTAL"],
    errors="coerce"
)

mask_total_coma_inicial = (
    total_auditoria_inicial.isna()
    &
    df["TOTAL"]
    .astype("string")
    .str.strip()
    .str.match(r"^-?\d+,\d+$", na=False)
)

if mask_total_coma_inicial.any():

    total_auditoria_inicial.loc[
        mask_total_coma_inicial
    ] = pd.to_numeric(
        df.loc[
            mask_total_coma_inicial,
            "TOTAL"
        ]
        .astype("string")
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )


TOTAL_ANTES_BLOQUE_4 = float(
    total_auditoria_inicial.sum()
)


print("=" * 75)
print("INICIO DEL BLOQUE 4")
print("=" * 75)

print(
    f"Filas iniciales: "
    f"{FILAS_ANTES_BLOQUE_4:,}"
)

print(
    f"Total inicial: "
    f"${TOTAL_ANTES_BLOQUE_4:,.2f}"
)

print("✅ df fue reconstruido desde consumos_original.")
print("✅ consumos_original permanece intacta.")


# =========================================================
# 2. FUNCIONES DE NORMALIZACIÓN
# =========================================================

def strip_accents(texto: str) -> str:
    """
    Elimina tildes y caracteres diacríticos.
    """

    return "".join(
        caracter
        for caracter in unicodedata.normalize(
            "NFKD",
            texto
        )
        if not unicodedata.combining(caracter)
    )


def norm_text(valor):
    """
    Normaliza texto sin convertir valores vacíos
    en textos como NAN o NONE.
    """

    if pd.isna(valor):
        return np.nan

    texto = str(valor)
    texto = strip_accents(texto)
    texto = texto.upper().strip()
    texto = re.sub(r"[_]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto)

    if texto in {
        "",
        "NAN",
        "NONE",
        "<NA>"
    }:
        return np.nan

    return texto


# =========================================================
# 3. NORMALIZAR CENTRO DE COSTO Y ÁREA
# =========================================================

df["CENTRO DE COSTO"] = (
    df["CENTRO DE COSTO"]
    .map(norm_text)
)


df["AREA"] = (
    df["AREA"]
    .map(norm_text)
)


mask_centro_vacio_inicial = (
    df["CENTRO DE COSTO"].isna()
    |
    df["CENTRO DE COSTO"]
    .astype("string")
    .str.strip()
    .fillna("")
    .eq("")
)


mask_area_vacia_inicial = (
    df["AREA"].isna()
    |
    df["AREA"]
    .astype("string")
    .str.strip()
    .fillna("")
    .eq("")
)


CENTROS_VACIOS_INICIALES = int(
    mask_centro_vacio_inicial.sum()
)


AREAS_VACIAS_INICIALES = int(
    mask_area_vacia_inicial.sum()
)


print("\n" + "=" * 75)
print("NORMALIZACIÓN INICIAL")
print("=" * 75)

print(
    f"Centros de costo vacíos: "
    f"{CENTROS_VACIOS_INICIALES:,}"
)

print(
    f"Áreas vacías antes de corregir: "
    f"{AREAS_VACIAS_INICIALES:,}"
)


# =========================================================
# 4. CREAR FECHA, AÑO, SEMANA Y TOTAL NUMÉRICO
# =========================================================

fecha_numerica = pd.to_numeric(
    df["FECHA"],
    errors="coerce"
)

fecha_desde_excel = pd.to_datetime(
    fecha_numerica,
    unit="D",
    origin="1899-12-30",
    errors="coerce"
)

fecha_desde_texto = pd.to_datetime(
    df["FECHA"].where(
        fecha_numerica.isna()
    ),
    errors="coerce"
)

df["_FECHA_DT"] = (
    fecha_desde_excel
    .fillna(fecha_desde_texto)
)


mask_fecha_invalida = (
    df["_FECHA_DT"].isna()
)


if mask_fecha_invalida.any():

    display(
        df.loc[
            mask_fecha_invalida,
            [
                "BODEGA",
                "FECHA",
                "SEMANA",
                "CENTRO DE COSTO",
                "TOTAL"
            ]
        ].head(100)
    )

    raise ValueError(
        "Existen fechas inválidas en la base de consumos."
    )


calendario_df = (
    df["_FECHA_DT"]
    .dt
    .isocalendar()
)


df["_AÑO_CRUCE"] = (
    calendario_df["year"]
    .astype("Int64")
)


df["_SEMANA_CRUCE"] = (
    calendario_df["week"]
    .astype("Int64")
)


df["_TOTAL_NUM"] = pd.to_numeric(
    df["TOTAL"],
    errors="coerce"
)

mask_total_coma = (
    df["_TOTAL_NUM"].isna()
    &
    df["TOTAL"]
    .astype("string")
    .str.strip()
    .str.match(r"^-?\d+,\d+$", na=False)
)

if mask_total_coma.any():

    df.loc[
        mask_total_coma,
        "_TOTAL_NUM"
    ] = pd.to_numeric(
        df.loc[
            mask_total_coma,
            "TOTAL"
        ]
        .astype("string")
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )


mask_total_invalido = (
    df["_TOTAL_NUM"].isna()
    &
    df["TOTAL"].notna()
    &
    df["TOTAL"]
    .astype("string")
    .str.strip()
    .ne("")
)


if mask_total_invalido.any():

    display(
        df.loc[
            mask_total_invalido,
            [
                "BODEGA",
                "FECHA",
                "SEMANA",
                "CENTRO DE COSTO",
                "TOTAL"
            ]
        ].head(100)
    )

    raise ValueError(
        "Existen valores de TOTAL que no pudieron "
        "convertirse a número."
    )


# =========================================================
# 5. CREAR FAMILIA_DESGLOSE INICIAL
# =========================================================

df["FAMILIA_DESGLOSE"] = (
    df["CENTRO DE COSTO"]
    .str.split()
    .str[0]
)


family_alias = {

    # GYPSOPHILA
    "GYPSO": "GYPSOPHILA",
    "GYPSOPHILA": "GYPSOPHILA",
    "GYPSOPHILIA": "GYPSOPHILA",
    "SMALL": "GYPSOPHILA",

    # SUNFLOWER
    "GIRASOL": "SUNFLOWER",
    "SUNFLOWER": "SUNFLOWER",

    # VERONICAS
    "VERONICA": "VERONICAS",
    "VERONICAS": "VERONICAS",

    # SOLIDAGO
    "SOLIDADO": "SOLIDAGO",
    "SOLIDAGO": "SOLIDAGO"
}


df["FAMILIA_DESGLOSE"] = (
    df["FAMILIA_DESGLOSE"]
    .replace(family_alias)
)


# =========================================================
# 6. HOMOLOGACIÓN ROBUSTA DESDE CENTRO DE COSTO
# =========================================================
#
# La clasificación no dependerá solamente de la primera
# palabra. Se revisa todo el CENTRO DE COSTO.
# =========================================================

centro_normalizado = (
    df["CENTRO DE COSTO"]
    .astype("string")
)


# ---------------------------------------------------------
# GYPSOPHILA
# ---------------------------------------------------------

mask_familia_gypsophila = (
    centro_normalizado
    .str.contains(
        (
            r"\bGYPSOPHILA\b"
            r"|\bGYPSOPHILIA\b"
            r"|\bGYPSO\b"
            r"|\bSMALL\s+BLOOM\b"
        ),
        regex=True,
        na=False
    )
)


df.loc[
    mask_familia_gypsophila,
    "FAMILIA_DESGLOSE"
] = "GYPSOPHILA"


# ---------------------------------------------------------
# SOLIDAGO / SOLIDADO
# ---------------------------------------------------------

mask_familia_solidago = (
    centro_normalizado
    .str.contains(
        r"\bSOLIDAGO\b|\bSOLIDADO\b",
        regex=True,
        na=False
    )
)


df.loc[
    mask_familia_solidago,
    "FAMILIA_DESGLOSE"
] = "SOLIDAGO"


# ---------------------------------------------------------
# VERONICAS
# ---------------------------------------------------------

mask_familia_veronicas = (
    centro_normalizado
    .str.contains(
        r"\bVERONICA\b|\bVERONICAS\b",
        regex=True,
        na=False
    )
)


df.loc[
    mask_familia_veronicas,
    "FAMILIA_DESGLOSE"
] = "VERONICAS"


# ---------------------------------------------------------
# SUNFLOWER
# ---------------------------------------------------------

mask_familia_sunflower = (
    centro_normalizado
    .str.contains(
        r"\bSUNFLOWER\b|\bGIRASOL\b",
        regex=True,
        na=False
    )
)


df.loc[
    mask_familia_sunflower,
    "FAMILIA_DESGLOSE"
] = "SUNFLOWER"


# =========================================================
# 7. CREAR FAMILIA FINAL
# =========================================================

familias_a_otros = {
    "BONSAI",
    "BOUQUETS",
    "BRILLANTINA",
    "CALAS",
    "COCULUS",
    "ENSAYOS",
    "HORTALIZAS",
    "JARDINES",
    "LIATRIS",
    "MATERIALES",
    "OTROS",
    "PRODUCTOS",
    "RUMEX",
    "SUCULENTAS",
    "WAX FLOWER",
    "BLUPLEURUM",
    "EUCALIPTO",
    "RICE",
    "RUSCUS",
    "MINDO",
    "ASTRANTIAS",
    "CRISANTEMO",
    "ARANDANOS",
    "ACHILLEA",
    "ACHILLEAS",
    "KALANCHOES"
}


df["FAMILIA"] = np.where(
    df["FAMILIA_DESGLOSE"]
    .isin(familias_a_otros),
    "OTROS",
    df["FAMILIA_DESGLOSE"]
)


# Refuerzo definitivo.
df["FAMILIA"] = (
    df["FAMILIA"]
    .replace({
        "GYPSO": "GYPSOPHILA",
        "GYPSOPHILA": "GYPSOPHILA",
        "GYPSOPHILIA": "GYPSOPHILA",
        "SMALL": "GYPSOPHILA",

        "SOLIDADO": "SOLIDAGO",
        "SOLIDAGO": "SOLIDAGO",

        "VERONICA": "VERONICAS",
        "VERONICAS": "VERONICAS",

        "GIRASOL": "SUNFLOWER"
    })
)


# =========================================================
# 8. DETERMINAR EL ÁREA DESDE CENTRO DE COSTO
# =========================================================
#
# A diferencia del bloque anterior:
#
# - No solamente llena áreas vacías.
# - También corrige áreas incorrectas.
#
# Ejemplo real:
#
# CENTRO DE COSTO:
# SOLIDADO PYGAN CIF PROPAGACION
#
# AREA original:
# ADMINISTRACION
#
# AREA corregida:
# PROPAGACION
# =========================================================

mask_centro_productivo = (
    centro_normalizado
    .str.contains(
        r"\bPRODUCTIVO\b|\bPRODUCTIVA\b",
        regex=True,
        na=False
    )
)


mask_centro_postcosecha = (
    centro_normalizado
    .str.contains(
        r"\bPOSTCOSECHA\b",
        regex=True,
        na=False
    )
)


mask_centro_propagacion = (
    centro_normalizado
    .str.contains(
        r"\bPROPAGACION\b",
        regex=True,
        na=False
    )
)


mask_centro_administracion = (
    centro_normalizado
    .str.contains(
        r"\bADMINISTRACION\b|\bADMINISTRATIVO\b",
        regex=True,
        na=False
    )
)


# =========================================================
# 8.1 VALIDAR CENTROS CON MÁS DE UN ÁREA
# =========================================================

cantidad_areas_detectadas = (
    pd.concat(
        [
            mask_centro_productivo,
            mask_centro_postcosecha,
            mask_centro_propagacion,
            mask_centro_administracion
        ],
        axis=1
    )
    .astype(int)
    .sum(axis=1)
)


mask_centro_area_ambigua = (
    cantidad_areas_detectadas > 1
)


if mask_centro_area_ambigua.any():

    print(
        "\n⚠️ CENTROS DE COSTO CON MÁS DE UN ÁREA:"
    )

    display(
        df.loc[
            mask_centro_area_ambigua,
            [
                "BODEGA",
                "FECHA",
                "SEMANA",
                "CENTRO DE COSTO",
                "AREA",
                "TOTAL"
            ]
        ].head(200)
    )

    raise ValueError(
        "Existen centros de costo que contienen "
        "más de una palabra de área."
    )


# =========================================================
# 8.2 CONSTRUIR ÁREA CORRECTA
# =========================================================

area_desde_centro = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string"
)


area_desde_centro.loc[
    mask_centro_productivo
] = "PRODUCTIVO"


area_desde_centro.loc[
    mask_centro_postcosecha
] = "POSTCOSECHA"


area_desde_centro.loc[
    mask_centro_propagacion
] = "PROPAGACION"


area_desde_centro.loc[
    mask_centro_administracion
] = "ADMINISTRACION"


area_original_normalizada = (
    df["AREA"]
    .map(norm_text)
    .astype("string")
)


mask_area_completada = (
    area_original_normalizada.isna()
    &
    area_desde_centro.notna()
)


mask_area_corregida_por_conflicto = (
    area_original_normalizada.notna()
    &
    area_desde_centro.notna()
    &
    area_original_normalizada
    .ne(area_desde_centro)
    .fillna(False)
)


mask_area_identificable = (
    area_desde_centro.notna()
)


# Cuando el centro identifica el área,
# CENTRO DE COSTO será la fuente de verdad.
df.loc[
    mask_area_identificable,
    "AREA"
] = (
    area_desde_centro.loc[
        mask_area_identificable
    ]
    .astype(object)
)


REGISTROS_AREA_COMPLETADOS = int(
    mask_area_completada.sum()
)


REGISTROS_AREA_CONFLICTO_CORREGIDOS = int(
    mask_area_corregida_por_conflicto.sum()
)


TOTAL_REGISTROS_AREA_MODIFICADOS = (
    REGISTROS_AREA_COMPLETADOS
    +
    REGISTROS_AREA_CONFLICTO_CORREGIDOS
)


print("\n" + "=" * 75)
print("CORRECCIÓN DEFINITIVA DE ÁREAS")
print("=" * 75)

print(
    f"Áreas vacías completadas: "
    f"{REGISTROS_AREA_COMPLETADOS:,}"
)

print(
    f"Áreas incorrectas corregidas: "
    f"{REGISTROS_AREA_CONFLICTO_CORREGIDOS:,}"
)

print(
    f"Total de registros modificados: "
    f"{TOTAL_REGISTROS_AREA_MODIFICADOS:,}"
)


# =========================================================
# 8.3 MOSTRAR ÁREAS QUE FUERON CORREGIDAS POR CONFLICTO
# =========================================================

if mask_area_corregida_por_conflicto.any():

    print(
        "\nÁREAS EXISTENTES QUE CONTRADECÍAN "
        "EL CENTRO DE COSTO:"
    )

    control_conflictos_area = (
        pd.DataFrame({
            "BODEGA": (
                df.loc[
                    mask_area_corregida_por_conflicto,
                    "BODEGA"
                ]
            ),
            "CENTRO DE COSTO": (
                df.loc[
                    mask_area_corregida_por_conflicto,
                    "CENTRO DE COSTO"
                ]
            ),
            "AREA_ORIGINAL": (
                area_original_normalizada.loc[
                    mask_area_corregida_por_conflicto
                ]
            ),
            "AREA_CORREGIDA": (
                df.loc[
                    mask_area_corregida_por_conflicto,
                    "AREA"
                ]
            ),
            "FAMILIA": (
                df.loc[
                    mask_area_corregida_por_conflicto,
                    "FAMILIA"
                ]
            ),
            "TOTAL": (
                df.loc[
                    mask_area_corregida_por_conflicto,
                    "_TOTAL_NUM"
                ]
            )
        })
        .groupby(
            [
                "BODEGA",
                "CENTRO DE COSTO",
                "AREA_ORIGINAL",
                "AREA_CORREGIDA",
                "FAMILIA"
            ],
            dropna=False
        )
        .agg(
            REGISTROS=("TOTAL", "size"),
            TOTAL=("TOTAL", "sum")
        )
        .reset_index()
    )

    display(control_conflictos_area)


# =========================================================
# 8.4 VALIDAR QUE LA CORRECCIÓN FUNCIONÓ
# =========================================================

area_final_normalizada = (
    df["AREA"]
    .map(norm_text)
    .astype("string")
)


mask_area_incorrecta_despues = (
    area_desde_centro.notna()
    &
    area_final_normalizada
    .ne(area_desde_centro)
    .fillna(True)
)


if mask_area_incorrecta_despues.any():

    display(
        df.loc[
            mask_area_incorrecta_despues,
            [
                "BODEGA",
                "FECHA",
                "SEMANA",
                "CENTRO DE COSTO",
                "FAMILIA",
                "AREA",
                "TOTAL"
            ]
        ].head(200)
    )

    raise ValueError(
        "Todavía existen áreas que no coinciden "
        "con el CENTRO DE COSTO."
    )


print(
    "✅ Todas las áreas identificables coinciden "
    "con CENTRO DE COSTO."
)


# =========================================================
# 8.5 REGISTROS QUE NO PERMITEN DETERMINAR EL ÁREA
# =========================================================

mask_area_no_identificable_vacia = (
    area_desde_centro.isna()
    &
    (
        df["AREA"].isna()
        |
        df["AREA"]
        .astype("string")
        .str.strip()
        .fillna("")
        .eq("")
    )
)


CANTIDAD_AREA_NO_IDENTIFICABLE = int(
    mask_area_no_identificable_vacia.sum()
)


print(
    f"Registros que continúan con AREA vacía porque "
    f"el centro no permite identificarla: "
    f"{CANTIDAD_AREA_NO_IDENTIFICABLE:,}"
)


if CANTIDAD_AREA_NO_IDENTIFICABLE > 0:

    resumen_area_no_identificable = (
        df.loc[
            mask_area_no_identificable_vacia
        ]
        .groupby(
            [
                "BODEGA",
                "CENTRO DE COSTO",
                "FAMILIA",
                "GRUPOS DE MATERIALES"
            ],
            dropna=False
        )
        .agg(
            REGISTROS=("TOTAL", "size"),
            TOTAL=("_TOTAL_NUM", "sum")
        )
        .reset_index()
        .sort_values(
            "TOTAL",
            ascending=False
        )
    )

    display(resumen_area_no_identificable)


# =========================================================
# 9. DEVOLUCIONES SIN CENTRO DE COSTO
# =========================================================

mask_centro_vacio_global = (
    df["CENTRO DE COSTO"].isna()
    |
    df["CENTRO DE COSTO"]
    .astype("string")
    .str.strip()
    .fillna("")
    .eq("")
)


mask_devolucion_sin_centro_global = (
    mask_centro_vacio_global
    &
    df["_TOTAL_NUM"]
    .fillna(0)
    .le(0)
)


mask_positivo_sin_centro_global = (
    mask_centro_vacio_global
    &
    df["_TOTAL_NUM"]
    .gt(0)
)


df.loc[
    mask_devolucion_sin_centro_global,
    "FAMILIA_DESGLOSE"
] = "SIN CENTRO DE COSTO"


df.loc[
    mask_devolucion_sin_centro_global,
    "FAMILIA"
] = "OTROS"


print("\n" + "=" * 75)
print("DEVOLUCIONES SIN CENTRO DE COSTO")
print("=" * 75)

print(
    f"Devoluciones sin centro aceptadas: "
    f"{mask_devolucion_sin_centro_global.sum():,}"
)

print(
    f"Consumos positivos sin centro: "
    f"{mask_positivo_sin_centro_global.sum():,}"
)

print(
    f"Total de devoluciones clasificadas como OTROS: "
    f"${df.loc[mask_devolucion_sin_centro_global, '_TOTAL_NUM'].sum():,.2f}"
)


# =========================================================
# 10. EXTRAER LA SEMANA PROCESADA
# =========================================================

mask_df_semana_proceso = (
    (df["_AÑO_CRUCE"] == ANIO_PROCESO)
    &
    (df["_SEMANA_CRUCE"] == SEMANA_PROCESO)
)


df_semana_proceso = (
    df.loc[
        mask_df_semana_proceso
    ]
    .copy()
)


print("\n" + "=" * 75)
print(
    f"SEMANA PROCESADA: "
    f"{ANIO_PROCESO} - {SEMANA_PROCESO}"
)
print("=" * 75)

print(
    f"Registros encontrados: "
    f"{len(df_semana_proceso):,}"
)

print(
    f"Total encontrado: "
    f"${df_semana_proceso['_TOTAL_NUM'].sum():,.2f}"
)


# =========================================================
# 11. VALIDAR CENTROS VACÍOS DE LA SEMANA
# =========================================================

mask_centro_vacio_semana = (
    df_semana_proceso["CENTRO DE COSTO"].isna()
    |
    df_semana_proceso["CENTRO DE COSTO"]
    .astype("string")
    .str.strip()
    .fillna("")
    .eq("")
)


mask_devolucion_sin_centro_semana = (
    mask_centro_vacio_semana
    &
    df_semana_proceso["_TOTAL_NUM"]
    .fillna(0)
    .le(0)
)


mask_positivo_sin_centro_semana = (
    mask_centro_vacio_semana
    &
    df_semana_proceso["_TOTAL_NUM"]
    .gt(0)
)


if mask_positivo_sin_centro_semana.any():

    print(
        "\n⚠️ CONSUMOS POSITIVOS SIN CENTRO DE COSTO:"
    )

    display(
        df_semana_proceso.loc[
            mask_positivo_sin_centro_semana,
            [
                "BODEGA",
                "FECHA",
                "SEMANA",
                "CENTRO DE COSTO",
                "NOMBRE",
                "TOTAL",
                "USUARIO"
            ]
        ]
        .sort_values(
            "TOTAL",
            ascending=False
        )
        .head(200)
    )

    raise ValueError(
        "Existen consumos positivos sin CENTRO DE COSTO."
    )


# =========================================================
# 12. VALIDAR FAMILIAS VACÍAS
# =========================================================

mask_familia_vacia_semana = (
    df_semana_proceso["FAMILIA"].isna()
    |
    df_semana_proceso["FAMILIA"]
    .astype("string")
    .str.strip()
    .fillna("")
    .eq("")
)


if mask_familia_vacia_semana.any():

    display(
        df_semana_proceso.loc[
            mask_familia_vacia_semana,
            [
                "BODEGA",
                "FECHA",
                "SEMANA",
                "CENTRO DE COSTO",
                "FAMILIA_DESGLOSE",
                "FAMILIA",
                "TOTAL"
            ]
        ].head(200)
    )

    raise ValueError(
        "Todavía existen registros sin familia "
        "en la semana procesada."
    )


# =========================================================
# 13. CONTROL DEFINITIVO DE GYPSOPHILA
# =========================================================

mask_centro_gyps_semana = (
    df_semana_proceso["CENTRO DE COSTO"]
    .astype("string")
    .str.contains(
        (
            r"\bGYPSOPHILA\b"
            r"|\bGYPSOPHILIA\b"
            r"|\bGYPSO\b"
            r"|\bSMALL\s+BLOOM\b"
        ),
        regex=True,
        na=False
    )
)


mask_gyps_clasificacion_incorrecta = (
    mask_centro_gyps_semana
    &
    ~df_semana_proceso["FAMILIA"]
    .eq("GYPSOPHILA")
)


if mask_gyps_clasificacion_incorrecta.any():

    display(
        df_semana_proceso.loc[
            mask_gyps_clasificacion_incorrecta,
            [
                "BODEGA",
                "CENTRO DE COSTO",
                "AREA",
                "FAMILIA_DESGLOSE",
                "FAMILIA",
                "GRUPOS DE MATERIALES",
                "TOTAL"
            ]
        ].head(300)
    )

    raise ValueError(
        "Todavía existen registros GYPSOPHILA o "
        "SMALL BLOOM que no quedaron como GYPSOPHILA."
    )


variantes_gyps_finales = set(
    df_semana_proceso.loc[
        mask_centro_gyps_semana,
        "FAMILIA"
    ]
    .dropna()
    .unique()
    .tolist()
)


if variantes_gyps_finales != {"GYPSOPHILA"}:

    raise ValueError(
        f"Se encontraron familias inesperadas para "
        f"Gypsophila: {sorted(variantes_gyps_finales)}"
    )


print(
    "✅ GYPSOPHILA, GYPSO y SMALL BLOOM "
    "quedaron unificados como GYPSOPHILA."
)


# =========================================================
# 14. CONTROL DEFINITIVO DE SOLIDAGO
# =========================================================

mask_centro_solidago_semana = (
    df_semana_proceso["CENTRO DE COSTO"]
    .astype("string")
    .str.contains(
        r"\bSOLIDAGO\b|\bSOLIDADO\b",
        regex=True,
        na=False
    )
)


mask_solidago_clasificacion_incorrecta = (
    mask_centro_solidago_semana
    &
    ~df_semana_proceso["FAMILIA"]
    .eq("SOLIDAGO")
)


if mask_solidago_clasificacion_incorrecta.any():

    display(
        df_semana_proceso.loc[
            mask_solidago_clasificacion_incorrecta,
            [
                "BODEGA",
                "CENTRO DE COSTO",
                "AREA",
                "FAMILIA_DESGLOSE",
                "FAMILIA",
                "GRUPOS DE MATERIALES",
                "TOTAL"
            ]
        ].head(300)
    )

    raise ValueError(
        "Todavía existen registros SOLIDAGO o SOLIDADO "
        "que no quedaron como SOLIDAGO."
    )


variantes_solidago_finales = set(
    df_semana_proceso.loc[
        mask_centro_solidago_semana,
        "FAMILIA"
    ]
    .dropna()
    .unique()
    .tolist()
)


if variantes_solidago_finales != {"SOLIDAGO"}:

    raise ValueError(
        f"Se encontraron familias inesperadas para "
        f"Solidago: {sorted(variantes_solidago_finales)}"
    )


print(
    "✅ SOLIDAGO y SOLIDADO quedaron unificados "
    "como SOLIDAGO."
)


# =========================================================
# 15. CONTROL DE PRODUCTIVO
# =========================================================

mask_productivo_semana = (
    df_semana_proceso["CENTRO DE COSTO"]
    .astype("string")
    .str.contains(
        r"\bPRODUCTIVO\b|\bPRODUCTIVA\b",
        regex=True,
        na=False
    )
)


mask_productivo_area_incorrecta = (
    mask_productivo_semana
    &
    ~df_semana_proceso["AREA"]
    .eq("PRODUCTIVO")
)


if mask_productivo_area_incorrecta.any():

    display(
        df_semana_proceso.loc[
            mask_productivo_area_incorrecta,
            [
                "BODEGA",
                "CENTRO DE COSTO",
                "FAMILIA",
                "AREA",
                "GRUPOS DE MATERIALES",
                "TOTAL"
            ]
        ].head(300)
    )

    raise ValueError(
        "Existen registros productivos cuyo AREA "
        "no quedó como PRODUCTIVO."
    )


print(
    "✅ Todos los centros productivos tienen "
    "AREA = PRODUCTIVO."
)


# =========================================================
# 16. RESUMEN DE FAMILIAS CLAVE
# =========================================================

familias_control = [
    "GYPSOPHILA",
    "HYPERICUM",
    "SOLIDAGO",
    "VERONICAS"
]


control_familias_clave = (
    df_semana_proceso.loc[
        df_semana_proceso["FAMILIA"]
        .isin(familias_control)
    ]
    .groupby(
        [
            "FAMILIA",
            "AREA",
            "GRUPOS DE MATERIALES"
        ],
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("_TOTAL_NUM", "sum")
    )
    .reset_index()
    .sort_values(
        [
            "FAMILIA",
            "AREA",
            "GRUPOS DE MATERIALES"
        ]
    )
)


print("\n" + "=" * 75)
print(
    f"CONTROL DE FAMILIAS CLAVE — "
    f"SEMANA {SEMANA_PROCESO}"
)
print("=" * 75)

display(control_familias_clave)


# =========================================================
# 17. AUDITAR OTROS
# =========================================================

otros_semana = (
    df_semana_proceso.loc[
        df_semana_proceso["FAMILIA"]
        .eq("OTROS")
    ]
    .copy()
)


resumen_otros_semana = (
    otros_semana
    .groupby(
        "FAMILIA_DESGLOSE",
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("_TOTAL_NUM", "sum")
    )
    .reset_index()
    .sort_values(
        "TOTAL",
        ascending=False
    )
)


print("\n" + "=" * 75)
print("DESGLOSE DE LA FAMILIA OTROS")
print("=" * 75)

print(
    f"Registros OTROS: "
    f"{len(otros_semana):,}"
)

print(
    f"Total OTROS: "
    f"${otros_semana['_TOTAL_NUM'].sum():,.2f}"
)

display(resumen_otros_semana)


# =========================================================
# 18. RESUMEN FINAL POR FAMILIA
# =========================================================

resumen_familias_semana = (
    df_semana_proceso
    .groupby(
        "FAMILIA",
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("_TOTAL_NUM", "sum")
    )
    .reset_index()
    .sort_values(
        "TOTAL",
        ascending=False
    )
)


print("\n" + "=" * 75)
print("RESUMEN FINAL POR FAMILIA")
print("=" * 75)

display(resumen_familias_semana)


# =========================================================
# 19. VALIDACIONES DE INTEGRIDAD
# =========================================================

FILAS_DESPUES_BLOQUE_4 = len(df)


TOTAL_DESPUES_BLOQUE_4 = float(
    df["_TOTAL_NUM"].sum()
)


if FILAS_DESPUES_BLOQUE_4 != FILAS_ANTES_BLOQUE_4:

    raise ValueError(
        "El bloque 4 cambió la cantidad de filas."
    )


if not np.isclose(
    TOTAL_ANTES_BLOQUE_4,
    TOTAL_DESPUES_BLOQUE_4,
    atol=0.01
):

    raise ValueError(
        "El bloque 4 cambió el total de consumos."
    )


if len(df_semana_proceso) != REGISTROS_SEMANA_PROCESO:

    raise ValueError(
        "La cantidad de registros de la semana "
        "no coincide con el bloque 3."
    )


if not np.isclose(
    df_semana_proceso["_TOTAL_NUM"].sum(),
    TOTAL_SEMANA_PROCESO,
    atol=0.01
):

    raise ValueError(
        "El total de la semana procesada "
        "no coincide con el bloque 3."
    )


if list(consumos_original.columns) != headers_consumos_originales:

    raise ValueError(
        "consumos_original fue modificada accidentalmente."
    )


# =========================================================
# 20. RESULTADO FINAL
# =========================================================

print("\n" + "=" * 75)
print("RESULTADO DEL BLOQUE 4")
print("=" * 75)

print(
    f"Filas antes: "
    f"{FILAS_ANTES_BLOQUE_4:,}"
)

print(
    f"Filas después: "
    f"{FILAS_DESPUES_BLOQUE_4:,}"
)

print(
    f"Total antes: "
    f"${TOTAL_ANTES_BLOQUE_4:,.2f}"
)

print(
    f"Total después: "
    f"${TOTAL_DESPUES_BLOQUE_4:,.2f}"
)

print(
    f"Áreas vacías completadas: "
    f"{REGISTROS_AREA_COMPLETADOS:,}"
)

print(
    f"Áreas incorrectas corregidas: "
    f"{REGISTROS_AREA_CONFLICTO_CORREGIDOS:,}"
)

print(
    f"Áreas todavía no identificables: "
    f"{CANTIDAD_AREA_NO_IDENTIFICABLE:,}"
)

print(
    f"Registros GYPSOPHILA semana "
    f"{SEMANA_PROCESO}: "
    f"{df_semana_proceso['FAMILIA'].eq('GYPSOPHILA').sum():,}"
)

print(
    f"Registros SOLIDAGO semana "
    f"{SEMANA_PROCESO}: "
    f"{df_semana_proceso['FAMILIA'].eq('SOLIDAGO').sum():,}"
)

print(
    f"Registros HYPERICUM semana "
    f"{SEMANA_PROCESO}: "
    f"{df_semana_proceso['FAMILIA'].eq('HYPERICUM').sum():,}"
)

print(
    f"Registros VERONICAS semana "
    f"{SEMANA_PROCESO}: "
    f"{df_semana_proceso['FAMILIA'].eq('VERONICAS').sum():,}"
)


print("\n✅ GYPSOPHILA y SMALL BLOOM quedaron como GYPSOPHILA.")
print("✅ SOLIDAGO y SOLIDADO quedaron como SOLIDAGO.")
print("✅ Las áreas vacías fueron completadas.")
print("✅ Las áreas contradictorias fueron corregidas.")
print("✅ Todos los registros productivos tienen AREA = PRODUCTIVO.")
print("✅ No cambiaron las filas ni el total.")
print("✅ consumos_original permanece intacta.")
print("✅ BLOQUE 4 TERMINADO.")

INICIO DEL BLOQUE 4
Filas iniciales: 376,489
Total inicial: $11,145,472.00
✅ df fue reconstruido desde consumos_original.
✅ consumos_original permanece intacta.

NORMALIZACIÓN INICIAL
Centros de costo vacíos: 350
Áreas vacías antes de corregir: 0

CORRECCIÓN DEFINITIVA DE ÁREAS
Áreas vacías completadas: 0
Áreas incorrectas corregidas: 3
Total de registros modificados: 3

ÁREAS EXISTENTES QUE CONTRADECÍAN EL CENTRO DE COSTO:


,BODEGA,CENTRO DE COSTO,AREA_ORIGINAL,AREA_CORREGIDA,FAMILIA,REGISTROS,TOTAL
0,Pygan,SOLIDADO PYGAN CIF PROPAGACION,ADMINISTRACION,PROPAGACION,SOLIDAGO,3,12.03


✅ Todas las áreas identificables coinciden con CENTRO DE COSTO.
Registros que continúan con AREA vacía porque el centro no permite identificarla: 0

DEVOLUCIONES SIN CENTRO DE COSTO
Devoluciones sin centro aceptadas: 350
Consumos positivos sin centro: 0
Total de devoluciones clasificadas como OTROS: $-25,902.67

SEMANA PROCESADA: 2026 - 34
Registros encontrados: 12,042
Total encontrado: $336,487.66
✅ GYPSOPHILA, GYPSO y SMALL BLOOM quedaron unificados como GYPSOPHILA.
✅ SOLIDAGO y SOLIDADO quedaron unificados como SOLIDAGO.
✅ Todos los centros productivos tienen AREA = PRODUCTIVO.

CONTROL DE FAMILIAS CLAVE — SEMANA 34


,FAMILIA,AREA,GRUPOS DE MATERIALES,REGISTROS,TOTAL
0,GYPSOPHILA,POSTCOSECHA,ACEITES Y COMBUSTIBLES,10,250.98
1,GYPSOPHILA,POSTCOSECHA,ASEO LIMPIEZA,13,609.71
2,GYPSOPHILA,POSTCOSECHA,CAPUCHONES,72,13995.05
3,GYPSOPHILA,POSTCOSECHA,CARTON,1038,20710.04
4,GYPSOPHILA,POSTCOSECHA,ELEMENTOS DE SEGURIDAD INDUSTRIAL,37,778.08
...,...,...,...,...,...
131,VERONICAS,PRODUCTIVO,SOLUCIONES,2,4.80
132,VERONICAS,PRODUCTIVO,TINTES,1,7.27
133,VERONICAS,PROPAGACION,FERTILIZANTES,75,36.82
134,VERONICAS,PROPAGACION,FUMIGANTES Y PESTICIDAS,4,15.01



DESGLOSE DE LA FAMILIA OTROS
Registros OTROS: 1,604
Total OTROS: $11,880.53


,FAMILIA_DESGLOSE,REGISTROS,TOTAL
4,BOUQUETS,116,4636.66
2,ARANDANOS,146,2094.72
14,PRODUCTOS,343,1735.08
15,RICE,189,1402.76
16,RUMEX,206,467.34
6,CRISANTEMO,87,421.64
8,EUCALIPTO,143,304.66
13,OTROS,17,290.74
12,MINDO,5,274.94
11,MATERIALES,16,96.44



RESUMEN FINAL POR FAMILIA


,FAMILIA,REGISTROS,TOTAL
3,GYPSOPHILA,2810,189127.02
12,ROSAS,1536,35042.58
11,RANUNCULOS,662,17128.64
10,OTROS,1604,11880.53
7,LIRIOS,404,10876.61
14,SOLIDAGO,563,10494.45
16,SUNFLOWER,276,9444.40
5,LEPIDIUM,413,8407.31
0,CRASPEDIAS,419,7679.50
6,LIMONIUM,941,7540.47



RESULTADO DEL BLOQUE 4
Filas antes: 376,489
Filas después: 376,489
Total antes: $11,145,472.00
Total después: $11,145,472.00
Áreas vacías completadas: 0
Áreas incorrectas corregidas: 3
Áreas todavía no identificables: 0
Registros GYPSOPHILA semana 34: 2,810
Registros SOLIDAGO semana 34: 563
Registros HYPERICUM semana 34: 240
Registros VERONICAS semana 34: 302

✅ GYPSOPHILA y SMALL BLOOM quedaron como GYPSOPHILA.
✅ SOLIDAGO y SOLIDADO quedaron como SOLIDAGO.
✅ Las áreas vacías fueron completadas.
✅ Las áreas contradictorias fueron corregidas.
✅ Todos los registros productivos tienen AREA = PRODUCTIVO.
✅ No cambiaron las filas ni el total.
✅ consumos_original permanece intacta.
✅ BLOQUE 4 TERMINADO.


# BLOQUE 5

In [ ]:
# =========================================================
# BLOQUE 5 — HOMOLOGACIÓN DE BODEGAS
# =========================================================
#
# Resultado:
# - Conserva BODEGA_ORIGINAL.
# - Homologa BODEGA a:
#       MALCHINGUI
#       PYGAN
#       URCUQUI
# - Aplica la regla especial:
#       BODEGA_ORIGINAL = ROSAS -> URCUQUI
# - No cambia filas ni valores de TOTAL.
#
# Todavía NO realiza el merge con hectáreas.
# =========================================================


# =========================================================
# 1. VALIDAR QUE EL BLOQUE 4 EXISTA
# =========================================================

columnas_necesarias = [
    "BODEGA",
    "FECHA",
    "SEMANA",
    "TOTAL",
    "CENTRO DE COSTO",
    "FAMILIA_DESGLOSE",
    "FAMILIA"
]

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in df.columns
]

if faltantes:
    raise ValueError(
        f"Faltan columnas del bloque anterior: {faltantes}"
    )


FILAS_ANTES_BLOQUE_5 = len(df)

TOTAL_ANTES_BLOQUE_5 = pd.to_numeric(
    df["TOTAL"],
    errors="coerce"
).sum()


# =========================================================
# 2. CONSERVAR BODEGA ORIGINAL
# =========================================================

if "BODEGA_ORIGINAL" not in df.columns:
    df["BODEGA_ORIGINAL"] = df["BODEGA"].copy()

print("✅ Se creó BODEGA_ORIGINAL.")
print("✅ La procedencia original de cada consumo queda protegida.")


# =========================================================
# 3. FUNCIÓN DE HOMOLOGACIÓN DE FINCAS
# =========================================================

def map_finca(valor):
    """
    Homologa los nombres originales de bodega.

    La normalización:
    - Quita tildes.
    - Convierte a mayúsculas.
    - Convierte espacios y guiones bajos.
    """

    texto = norm_text(valor)

    if pd.isna(texto):
        return np.nan

    clave = texto.replace(" ", "_")

    bodegas_malchingui = {
        "PRINCIPAL",
        "BODEGA_PRINCIPAL",
        "FLORSANI_N3",
        "FLORSANI_N_3",
        "BODEGA_PRINCIPAL_N3",
        "BODEGA_PRINCIPAL_N_3",
        "MALCHINGUI"
    }

    bodegas_urcuqui = {
        "X_SECUANDARIA",      # Nombre que aparece en tu fuente
        "X_SECUNDARIA",
        "SECUANDARIA",
        "SECUNDARIA",
        "BODEGA_SECUANDARIA",
        "BODEGA_SECUNDARIA",
        "URCUQUI"
    }

    bodegas_pygan = {
        "PYGAN",
        "PYGAN",
        "BODEGA_PYGAN"
    }

    if clave in bodegas_malchingui:
        return "MALCHINGUI"

    if clave in bodegas_urcuqui:
        return "URCUQUI"

    if clave in bodegas_pygan:
        return "PYGAN"

    # ROSAS se deja temporalmente identificable.
    # La regla especial se aplica después.
    if clave == "ROSAS":
        return "ROSAS"

    # No inventar una finca si aparece un nombre nuevo.
    return texto


# =========================================================
# 4. HOMOLOGAR BODEGA
# =========================================================

df["BODEGA"] = (
    df["BODEGA_ORIGINAL"]
    .map(map_finca)
)


# =========================================================
# 5. REGLA ESPECIAL DEL CÓDIGO ORIGINAL
# ROSAS -> URCUQUI
# =========================================================

mask_bodega_rosas = (
    df["BODEGA_ORIGINAL"]
    .map(norm_text)
    .eq("ROSAS")
)

REGISTROS_BODEGA_ROSAS = int(
    mask_bodega_rosas.sum()
)

TOTAL_BODEGA_ROSAS = float(
    pd.to_numeric(
        df.loc[mask_bodega_rosas, "TOTAL"],
        errors="coerce"
    ).sum()
)

df.loc[
    mask_bodega_rosas,
    "BODEGA"
] = "URCUQUI"

print("\n" + "=" * 75)
print("REGLA ESPECIAL: ROSAS → URCUQUI")
print("=" * 75)

print(
    f"Registros reasignados: "
    f"{REGISTROS_BODEGA_ROSAS:,}"
)

print(
    f"Total reasignado: "
    f"${TOTAL_BODEGA_ROSAS:,.2f}"
)


# =========================================================
# 6. AUDITAR LA HOMOLOGACIÓN
# =========================================================

tabla_homologacion_bodegas = (
    df[
        [
            "BODEGA_ORIGINAL",
            "BODEGA"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "BODEGA",
            "BODEGA_ORIGINAL"
        ]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 75)
print("TABLA DE HOMOLOGACIÓN")
print("=" * 75)

display(tabla_homologacion_bodegas)


# =========================================================
# 7. VALIDAR QUE SOLO QUEDEN TRES FINCAS
# =========================================================

bodegas_permitidas = {
    "MALCHINGUI",
    "PYGAN",
    "URCUQUI"
}

bodegas_encontradas = set(
    df["BODEGA"]
    .dropna()
    .unique()
    .tolist()
)

bodegas_no_reconocidas = (
    bodegas_encontradas
    - bodegas_permitidas
)

print("\nBodegas finales encontradas:")
print(sorted(bodegas_encontradas))

if bodegas_no_reconocidas:

    print("\n⚠️ Bodegas no homologadas:")
    print(sorted(bodegas_no_reconocidas))

    detalle_no_homologadas = (
        df.loc[
            df["BODEGA"].isin(bodegas_no_reconocidas),
            [
                "BODEGA_ORIGINAL",
                "BODEGA",
                "FECHA",
                "CENTRO DE COSTO",
                "TOTAL"
            ]
        ]
    )

    display(detalle_no_homologadas.head(100))

    raise ValueError(
        "Existen bodegas no homologadas. "
        "No se puede continuar al merge."
    )

print("✅ Solo quedaron MALCHINGUI, PYGAN y URCUQUI.")


# =========================================================
# 8. RESUMEN GLOBAL POR BODEGA
# =========================================================

resumen_bodegas_global = (
    df
    .groupby(
        "BODEGA",
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("_TOTAL_NUM", "sum")
    )
    .reset_index()
    .sort_values("BODEGA")
)

print("\n" + "=" * 75)
print("RESUMEN GLOBAL POR BODEGA")
print("=" * 75)

display(resumen_bodegas_global)


# =========================================================
# 9. RESUMEN DE LA SEMANA PROCESADA
# =========================================================

df_semana_proceso = (
    df.loc[
        (df["_AÑO_CRUCE"] == ANIO_PROCESO)
        &
        (df["_SEMANA_CRUCE"] == SEMANA_PROCESO)
    ]
    .copy()
)

resumen_bodegas_semana = (
    df_semana_proceso
    .groupby(
        "BODEGA",
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("_TOTAL_NUM", "sum")
    )
    .reset_index()
    .sort_values("BODEGA")
)

print("\n" + "=" * 75)
print(
    f"BODEGAS — {ANIO_PROCESO}, "
    f"SEMANA {SEMANA_PROCESO}"
)
print("=" * 75)

display(resumen_bodegas_semana)


# =========================================================
# 10. VALIDAR INTEGRIDAD
# =========================================================

FILAS_DESPUES_BLOQUE_5 = len(df)

TOTAL_DESPUES_BLOQUE_5 = pd.to_numeric(
    df["TOTAL"],
    errors="coerce"
).sum()

if FILAS_DESPUES_BLOQUE_5 != FILAS_ANTES_BLOQUE_5:
    raise ValueError(
        "La homologación de bodegas cambió "
        "la cantidad de registros."
    )

if not np.isclose(
    TOTAL_ANTES_BLOQUE_5,
    TOTAL_DESPUES_BLOQUE_5,
    atol=0.01
):
    raise ValueError(
        "La homologación cambió el total de consumos."
    )

if len(df_semana_proceso) != REGISTROS_SEMANA_PROCESO:
    raise ValueError(
        "Cambió la cantidad de registros de la semana procesada."
    )

if not np.isclose(
    df_semana_proceso["_TOTAL_NUM"].sum(),
    TOTAL_SEMANA_PROCESO,
    atol=0.01
):
    raise ValueError(
        "Cambió el total de la semana procesada."
    )


# =========================================================
# 11. RESULTADO
# =========================================================

print("\n" + "=" * 75)
print("RESULTADO DEL BLOQUE 5")
print("=" * 75)

print(
    f"Filas antes: "
    f"{FILAS_ANTES_BLOQUE_5:,}"
)

print(
    f"Filas después: "
    f"{FILAS_DESPUES_BLOQUE_5:,}"
)

print(
    f"Total antes: "
    f"${TOTAL_ANTES_BLOQUE_5:,.2f}"
)

print(
    f"Total después: "
    f"${TOTAL_DESPUES_BLOQUE_5:,.2f}"
)

print(
    f"Registros ROSAS reasignados a URCUQUI: "
    f"{REGISTROS_BODEGA_ROSAS:,}"
)

print("✅ BODEGA_ORIGINAL fue conservada.")
print("✅ BODEGA fue homologada.")
print("✅ ROSAS fue asignada a URCUQUI.")
print("✅ No cambiaron las filas ni el total.")
print("✅ BLOQUE 5 TERMINADO")
print("Todavía no se ha realizado el merge con hectáreas.")

✅ Se creó BODEGA_ORIGINAL.
✅ La procedencia original de cada consumo queda protegida.

REGLA ESPECIAL: ROSAS → URCUQUI
Registros reasignados: 1,165
Total reasignado: $26,762.49

TABLA DE HOMOLOGACIÓN


,BODEGA_ORIGINAL,BODEGA
0,Florsani N3,MALCHINGUI
1,Principal,MALCHINGUI
2,Pygan,PYGAN
3,Rosas,URCUQUI
4,Urcuqui,URCUQUI
5,x_Secuandaria,URCUQUI



Bodegas finales encontradas:
['MALCHINGUI', 'PYGAN', 'URCUQUI']
✅ Solo quedaron MALCHINGUI, PYGAN y URCUQUI.

RESUMEN GLOBAL POR BODEGA


,BODEGA,REGISTROS,TOTAL
0,MALCHINGUI,276561,9.037632e+06
1,PYGAN,21603,8.046893e+05
2,URCUQUI,78325,1.303151e+06



BODEGAS — 2026, SEMANA 34


,BODEGA,REGISTROS,TOTAL
0,MALCHINGUI,8662,264648.64
1,PYGAN,714,23127.99
2,URCUQUI,2666,48711.03



RESULTADO DEL BLOQUE 5
Filas antes: 376,489
Filas después: 376,489
Total antes: $11,145,469.18
Total después: $11,145,469.18
Registros ROSAS reasignados a URCUQUI: 1,165
✅ BODEGA_ORIGINAL fue conservada.
✅ BODEGA fue homologada.
✅ ROSAS fue asignada a URCUQUI.
✅ No cambiaron las filas ni el total.
✅ BLOQUE 5 TERMINADO
Todavía no se ha realizado el merge con hectáreas.


# BLOQUE 6

In [ ]:
# BLOQUE 6 — CRUCE DEFINITIVO Y PROTEGIDO CON HECTÁREAS
# =========================================================
#
# REGLAS:
#
# HYPERICUM  -> PYGAN
# VERONICAS  -> PYGAN
#
# LEPIDIUM   -> URCUQUI
# LISIANTHUS -> URCUQUI
# ROSAS      -> URCUQUI
# SUNFLOWER  -> URCUQUI
#
# GYPSOPHILA:
# - Si BODEGA = URCUQUI -> usar MALCHINGUI
# - Si BODEGA = PYGAN -> mantener PYGAN
# - Si BODEGA = MALCHINGUI -> mantener MALCHINGUI
#
# CORRECCIONES:
#
# - VERONICA y VERONICAS se unifican en consumos.
# - VERONICA y VERONICAS se unifican en hectáreas.
# - Se auditan explícitamente:
#       PYGAN + HYPERICUM
#       PYGAN + VERONICAS
#
# - Si falta una llave o no tiene hectáreas positivas,
#   el proceso se detiene antes del merge.
#
# - Si este bloque falla, el bloque 7 no podrá exportar
#   resultados viejos.
# =========================================================


# =========================================================
# 1. VALIDAR VARIABLES ANTERIORES
# =========================================================

variables_necesarias = [
    "df",
    "hectareas_original",
    "ANIO_PROCESO",
    "SEMANA_PROCESO",
    "REGISTROS_SEMANA_PROCESO",
    "TOTAL_SEMANA_PROCESO"
]


variables_faltantes = [
    variable
    for variable in variables_necesarias
    if variable not in globals()
]


if variables_faltantes:

    raise ValueError(
        f"Faltan variables de bloques anteriores: "
        f"{variables_faltantes}"
    )


columnas_consumos_necesarias = [
    "BODEGA_ORIGINAL",
    "BODEGA",
    "FECHA",
    "SEMANA",
    "TOTAL",
    "AREA",
    "FAMILIA_DESGLOSE",
    "FAMILIA",
    "_AÑO_CRUCE",
    "_SEMANA_CRUCE",
    "_TOTAL_NUM"
]


columnas_consumos_faltantes = [
    columna
    for columna in columnas_consumos_necesarias
    if columna not in df.columns
]


if columnas_consumos_faltantes:

    raise ValueError(
        f"Faltan columnas en df: "
        f"{columnas_consumos_faltantes}"
    )


columnas_hectareas_necesarias = [
    "AÑO",
    "SEMANA",
    "HECTARIAS",
    "FINCA",
    "FAMILIA"
]


columnas_hectareas_faltantes = [
    columna
    for columna in columnas_hectareas_necesarias
    if columna not in hectareas_original.columns
]


if columnas_hectareas_faltantes:

    raise ValueError(
        f"Faltan columnas en hectareas_original: "
        f"{columnas_hectareas_faltantes}"
    )


# =========================================================
# 2. FUNCIÓN PARA NORMALIZAR LLAVES
# =========================================================

def normalizar_llave_merge(valor):

    if pd.isna(valor):
        return np.nan

    texto = str(valor)

    texto = "".join(
        caracter
        for caracter in unicodedata.normalize(
            "NFKD",
            texto
        )
        if not unicodedata.combining(caracter)
    )

    texto = texto.upper().strip()
    texto = re.sub(r"[_]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto)

    return texto


# =========================================================
# 3. ELIMINAR RESULTADOS DE UN MERGE ANTERIOR
# =========================================================
#
# IMPORTANTE:
#
# Aquí se eliminan directamente de df las columnas creadas
# por una ejecución anterior del bloque 6.
#
# Si este bloque se detiene, el bloque 7 fallará porque ya
# no existirán BODEGA_HECTAREA, HECTARIAS ni
# COSTO_POR_HECTARIA.
#
# Así evitamos exportar resultados viejos.
# =========================================================

columnas_creadas_bloque_6 = [
    "BODEGA_HECTAREA",
    "_BODEGA_HECT_KEY",
    "_FAMILIA_KEY",
    "HECTARIAS",
    "COSTO_POR_HECTARIA",
    "_MATCH_HECTAREA"
]


df = (
    df
    .drop(
        columns=columnas_creadas_bloque_6,
        errors="ignore"
    )
    .copy()
)


# =========================================================
# 3.1 RECONSTRUIR AÑO Y SEMANA DESDE FECHA
# =========================================================
#
# IMPORTANTE:
# La columna SEMANA original puede venir como texto, por ejemplo:
# "18 Abril".
#
# Para el cruce con hectáreas NO dependemos de ese texto.
# La fuente de verdad temporal es FECHA y de allí obtenemos
# año ISO y semana ISO.
#
# Esto evita que registros históricos queden sin match de
# hectáreas por tener SEMANA en formato texto.
# =========================================================

# FECHA puede venir mezclada:
# - como serial de Excel (ej. 46029)
# - como fecha escrita (ej. 2026-01-08)
# - como Timestamp ya convertido
#
# Primero detectamos seriales Excel plausibles y luego
# convertimos el resto como fecha normal.

fecha_num_b6 = pd.to_numeric(
    df["FECHA"],
    errors="coerce"
)

mask_fecha_serial_excel_b6 = (
    fecha_num_b6.notna()
    & fecha_num_b6.between(20000, 80000)
)

fecha_cruce_b6 = pd.Series(
    pd.NaT,
    index=df.index,
    dtype="datetime64[ns]"
)

fecha_cruce_b6.loc[mask_fecha_serial_excel_b6] = pd.to_datetime(
    fecha_num_b6.loc[mask_fecha_serial_excel_b6],
    unit="D",
    origin="1899-12-30",
    errors="coerce"
)

mask_fecha_no_serial_b6 = ~mask_fecha_serial_excel_b6

fecha_cruce_b6.loc[mask_fecha_no_serial_b6] = pd.to_datetime(
    df.loc[mask_fecha_no_serial_b6, "FECHA"],
    errors="coerce"
)

mask_fecha_cruce_invalida = fecha_cruce_b6.isna()

if mask_fecha_cruce_invalida.any():

    display(
        df.loc[
            mask_fecha_cruce_invalida,
            [
                "FECHA",
                "SEMANA",
                "BODEGA",
                "FAMILIA",
                "TOTAL"
            ]
        ].head(100)
    )

    raise ValueError(
        "Existen fechas inválidas. No se puede reconstruir "
        "el año y la semana para el cruce con hectáreas."
    )


anio_cruce_anterior = pd.to_numeric(
    df["_AÑO_CRUCE"],
    errors="coerce"
).astype("Int64")

semana_cruce_anterior = pd.to_numeric(
    df["_SEMANA_CRUCE"],
    errors="coerce"
).astype("Int64")


calendario_cruce_b6 = (
    fecha_cruce_b6
    .dt
    .isocalendar()
)

anio_cruce_nuevo = (
    calendario_cruce_b6["year"]
    .astype("Int64")
)

semana_cruce_nueva = (
    calendario_cruce_b6["week"]
    .astype("Int64")
)


mask_periodo_reconstruido = (
    anio_cruce_anterior.isna()
    |
    semana_cruce_anterior.isna()
    |
    anio_cruce_anterior.ne(anio_cruce_nuevo).fillna(True)
    |
    semana_cruce_anterior.ne(semana_cruce_nueva).fillna(True)
)

REGISTROS_PERIODO_RECONSTRUIDO = int(
    mask_periodo_reconstruido.sum()
)


df["_AÑO_CRUCE"] = anio_cruce_nuevo
df["_SEMANA_CRUCE"] = semana_cruce_nueva


mask_periodo_invalido_b6 = (
    df["_AÑO_CRUCE"].isna()
    |
    df["_SEMANA_CRUCE"].isna()
    |
    ~df["_AÑO_CRUCE"].between(2000, 2100)
    |
    ~df["_SEMANA_CRUCE"].between(1, 53)
)


print("\n" + "=" * 75)
print("RECONSTRUCCIÓN DEL PERIODO PARA EL MERGE")
print("=" * 75)

print(
    f"Registros cuyo periodo fue reconstruido/corregido: "
    f"{REGISTROS_PERIODO_RECONSTRUIDO:,}"
)

print(
    f"Registros con periodo inválido después de reconstruir: "
    f"{int(mask_periodo_invalido_b6.sum()):,}"
)


if mask_periodo_invalido_b6.any():

    display(
        df.loc[
            mask_periodo_invalido_b6,
            [
                "FECHA",
                "SEMANA",
                "_AÑO_CRUCE",
                "_SEMANA_CRUCE",
                "BODEGA",
                "FAMILIA",
                "TOTAL"
            ]
        ].head(100)
    )

    raise ValueError(
        "Todavía existen registros con año o semana inválidos "
        "después de reconstruir el periodo desde FECHA."
    )


print("✅ Año y semana de cruce fueron reconstruidos desde FECHA.")
print("✅ Ningún registro queda con periodo inválido para el merge.")


df_base_merge = df.copy()


FILAS_ANTES_MERGE = len(df_base_merge)


TOTAL_ANTES_MERGE = float(
    pd.to_numeric(
        df_base_merge["TOTAL"],
        errors="coerce"
    ).sum()
)


print("=" * 75)
print("CONTROL ANTES DEL MERGE")
print("=" * 75)

print(
    f"Filas antes: "
    f"{FILAS_ANTES_MERGE:,}"
)

print(
    f"Total antes: "
    f"${TOTAL_ANTES_MERGE:,.2f}"
)


# =========================================================
# 4. PREPARAR BASE DE HECTÁREAS
# =========================================================

hect = hectareas_original.copy(deep=True)


hect["_AÑO_CRUCE"] = pd.to_numeric(
    hect["AÑO"],
    errors="coerce"
).astype("Int64")


# La semana de hectáreas debe ser numérica.
# También acepta accidentalmente texto como "31.0".
hect["_SEMANA_CRUCE"] = pd.to_numeric(
    hect["SEMANA"],
    errors="coerce"
).astype("Int64")


texto_hectarias = (
    hect["HECTARIAS"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False)
)


hect["_HECTARIAS_NUM"] = pd.to_numeric(
    texto_hectarias,
    errors="coerce"
)


hect["_BODEGA_HECT_KEY"] = (
    hect["FINCA"]
    .map(normalizar_llave_merge)
)


hect["_FAMILIA_KEY"] = (
    hect["FAMILIA"]
    .map(normalizar_llave_merge)
    .replace({
        "GYPSO": "GYPSOPHILA",
        "SMALL": "GYPSOPHILA",

        "GIRASOL": "SUNFLOWER",

        "SOLIDADO": "SOLIDAGO",

        # CORRECCIÓN VERÓNICAS
        "VERONICA": "VERONICAS",
        "VERONICAS": "VERONICAS"
    })
)


# =========================================================
# 5. VALIDAR VALORES DE HECTÁREAS
# =========================================================

mask_hectaria_con_contenido = (
    hect["HECTARIAS"].notna()
    &
    hect["HECTARIAS"]
    .astype("string")
    .str.strip()
    .ne("")
)


mask_texto_no_numerico = (
    mask_hectaria_con_contenido
    &
    hect["_HECTARIAS_NUM"].isna()
)


mask_hectaria_negativa = (
    hect["_HECTARIAS_NUM"] < 0
)


print("\n" + "=" * 75)
print("VALIDACIÓN DE VALORES DE HECTÁREAS")
print("=" * 75)

print(
    f"Valores positivos: "
    f"{(hect['_HECTARIAS_NUM'] > 0).sum():,}"
)

print(
    f"Valores en cero: "
    f"{(hect['_HECTARIAS_NUM'] == 0).sum():,}"
)

print(
    f"Valores vacíos: "
    f"{hect['_HECTARIAS_NUM'].isna().sum():,}"
)

print(
    f"Textos no numéricos: "
    f"{mask_texto_no_numerico.sum():,}"
)

print(
    f"Valores negativos: "
    f"{mask_hectaria_negativa.sum():,}"
)


if mask_texto_no_numerico.any():

    display(
        hect.loc[
            mask_texto_no_numerico,
            columnas_hectareas_necesarias
        ]
    )

    raise ValueError(
        "Existen textos no numéricos en HECTARIAS."
    )


if mask_hectaria_negativa.any():

    display(
        hect.loc[
            mask_hectaria_negativa,
            columnas_hectareas_necesarias
        ]
    )

    raise ValueError(
        "Existen valores negativos en HECTARIAS."
    )


print("✅ Los valores de hectáreas son válidos.")


# =========================================================
# 6. VALIDAR LLAVES DE HECTÁREAS
# =========================================================

mask_llave_invalida = (
    hect["_AÑO_CRUCE"].isna()
    |
    ~hect["_AÑO_CRUCE"].between(
        2000,
        2100
    )
    |
    hect["_SEMANA_CRUCE"].isna()
    |
    ~hect["_SEMANA_CRUCE"].between(
        1,
        53
    )
    |
    hect["_BODEGA_HECT_KEY"].isna()
    |
    hect["_BODEGA_HECT_KEY"].eq("")
    |
    hect["_FAMILIA_KEY"].isna()
    |
    hect["_FAMILIA_KEY"].eq("")
)


print("\n" + "=" * 75)
print("VALIDACIÓN DE LLAVES DE HECTÁREAS")
print("=" * 75)

print(
    f"Registros con llave inválida: "
    f"{mask_llave_invalida.sum():,}"
)


if mask_llave_invalida.any():

    display(
        hect.loc[
            mask_llave_invalida,
            columnas_hectareas_necesarias
        ]
    )

    raise ValueError(
        "Existen llaves inválidas en hectáreas."
    )


print("✅ Todas las llaves de hectáreas son válidas.")


# =========================================================
# 7. LLAVE ÚNICA DEL MERGE
# =========================================================

llaves_merge = [
    "_AÑO_CRUCE",
    "_SEMANA_CRUCE",
    "_BODEGA_HECT_KEY",
    "_FAMILIA_KEY"
]


# =========================================================
# 8. AUDITAR DUPLICADOS Y RESOLVER CON LA HECTÁREA MAYOR
# =========================================================
#
# REGLA DEFINITIVA:
# Si para la misma llave existen dos o más valores de
# hectáreas, NO se detiene el proceso.
#
# Se conserva automáticamente la HECTÁREA MÁS ALTA.
#
# Ejemplo:
# 2025 | 36 | URCUQUI | GYPSOPHILA | 4.98
# 2025 | 36 | URCUQUI | GYPSOPHILA | 1.78
#
# Resultado utilizado para el merge: 4.98
#
# Esto también resuelve correctamente casos con:
# - una fila vacía + una fila positiva;
# - duplicados idénticos;
# - duplicados con valores diferentes.
# =========================================================

conteo_valores_distintos = (
    hect
    .groupby(
        llaves_merge,
        dropna=False
    )["_HECTARIAS_NUM"]
    .nunique(dropna=True)
)

llaves_con_multiples_valores = (
    conteo_valores_distintos.loc[
        conteo_valores_distintos > 1
    ]
    .reset_index(
        name="VALORES_NUMERICOS_DISTINTOS"
    )
)

print("\n" + "=" * 75)
print("DUPLICADOS CON HECTÁREAS DIFERENTES")
print("=" * 75)

print(
    f"Llaves con más de un valor numérico: "
    f"{len(llaves_con_multiples_valores):,}"
)

if not llaves_con_multiples_valores.empty:

    detalle_conflictos = (
        hect
        .merge(
            llaves_con_multiples_valores,
            on=llaves_merge,
            how="inner"
        )
        [
            [
                "AÑO",
                "SEMANA",
                "FINCA",
                "FAMILIA",
                "HECTARIAS",
                "_BODEGA_HECT_KEY",
                "_FAMILIA_KEY",
                "VALORES_NUMERICOS_DISTINTOS"
            ]
        ]
        .sort_values(
            [
                "AÑO",
                "SEMANA",
                "_BODEGA_HECT_KEY",
                "_FAMILIA_KEY",
                "HECTARIAS"
            ],
            ascending=[True, True, True, True, False]
        )
    )

    print(
        "⚠️ Se encontraron valores distintos para una misma llave. "
        "El proceso continuará usando la HECTÁREA MÁS ALTA."
    )

    display(detalle_conflictos)

else:

    print("✅ No existen llaves con valores diferentes.")


# =========================================================
# 9. CONSOLIDAR TODAS LAS LLAVES USANDO EL VALOR MÁXIMO
# =========================================================

mask_duplicados = (
    hect
    .duplicated(
        subset=llaves_merge,
        keep=False
    )
)


duplicados_hectareas = (
    hect.loc[
        mask_duplicados,
        [
            "AÑO",
            "SEMANA",
            "FINCA",
            "FAMILIA",
            "HECTARIAS",
            "_AÑO_CRUCE",
            "_SEMANA_CRUCE",
            "_BODEGA_HECT_KEY",
            "_FAMILIA_KEY",
            "_HECTARIAS_NUM"
        ]
    ]
    .copy()
)


print("\n" + "=" * 75)
print("CONSOLIDACIÓN DE HECTÁREAS")
print("=" * 75)

print(
    f"Filas involucradas en llaves duplicadas: "
    f"{len(duplicados_hectareas):,}"
)


# Una sola fila por llave. Pandas max() ignora NaN cuando existe
# al menos un valor numérico, por lo que una fila vacía + una
# positiva conserva correctamente la positiva.
hectareas_merge = (
    hect
    .groupby(
        llaves_merge,
        as_index=False,
        dropna=False
    )
    .agg(
        HECTARIAS=(
            "_HECTARIAS_NUM",
            "max"
        )
    )
)


FILAS_HECTAREAS_ANTES = len(hect)
FILAS_HECTAREAS_DESPUES = len(hectareas_merge)

FILAS_REDUNDANTES_ELIMINADAS = (
    FILAS_HECTAREAS_ANTES
    - FILAS_HECTAREAS_DESPUES
)


print(
    f"Filas antes: "
    f"{FILAS_HECTAREAS_ANTES:,}"
)

print(
    f"Filas después: "
    f"{FILAS_HECTAREAS_DESPUES:,}"
)

print(
    f"Filas consolidadas: "
    f"{FILAS_REDUNDANTES_ELIMINADAS:,}"
)


if hectareas_merge.duplicated(
    subset=llaves_merge
).any():

    raise ValueError(
        "hectareas_merge todavía contiene duplicados."
    )


# Auditoría específica de las llaves donde había más de un valor.
if not llaves_con_multiples_valores.empty:

    control_valor_elegido = (
        llaves_con_multiples_valores
        .merge(
            hectareas_merge,
            on=llaves_merge,
            how="left"
        )
        .rename(
            columns={
                "HECTARIAS": "HECTARIA_ELEGIDA_MAX"
            }
        )
    )

    print("\nVALOR DEFINITIVO ELEGIDO POR LLAVE:")
    display(control_valor_elegido)


print("✅ hectareas_merge quedó única por llave.")
print("✅ Cuando existían varios valores, se conservó la hectárea más alta.")


# =========================================================
# 10. HOMOLOGAR VERÓNICAS EN CONSUMOS
# =========================================================

familia_consumo_normalizada = (
    df_base_merge["FAMILIA"]
    .map(normalizar_llave_merge)
)


mask_veronica_consumo = (
    familia_consumo_normalizada
    .isin([
        "VERONICA",
        "VERONICAS"
    ])
)


df_base_merge.loc[
    mask_veronica_consumo,
    "FAMILIA"
] = "VERONICAS"


desglose_consumo_normalizado = (
    df_base_merge["FAMILIA_DESGLOSE"]
    .map(normalizar_llave_merge)
)


mask_veronica_desglose = (
    desglose_consumo_normalizado
    .isin([
        "VERONICA",
        "VERONICAS"
    ])
)


df_base_merge.loc[
    mask_veronica_desglose,
    "FAMILIA_DESGLOSE"
] = "VERONICAS"


df_base_merge["_FAMILIA_KEY"] = (
    df_base_merge["FAMILIA"]
    .map(normalizar_llave_merge)
    .replace({
        "SOLIDADO": "SOLIDAGO",
        "VERONICA": "VERONICAS",
        "VERONICAS": "VERONICAS"
    })
)


# =========================================================
# 11. CREAR BODEGA_HECTAREA
# =========================================================

df_base_merge["BODEGA_HECTAREA"] = (
    df_base_merge["BODEGA"]
    .copy()
)


# =========================================================
# 12. MAPA DE FINCAS PRODUCTIVAS
# =========================================================

map_bodega_especial = {
    "HYPERICUM": "PYGAN",
    "VERONICAS": "PYGAN",

    "LEPIDIUM": "URCUQUI",
    "LISIANTHUS": "URCUQUI",
    "ROSAS": "URCUQUI",
    "SUNFLOWER": "URCUQUI",

    # Regla especial confirmada:
    # CRASPEDIAS usa las hectáreas de MALCHINGUI.
    "CRASPEDIAS": "MALCHINGUI"
}


mask_familia_especial = (
    df_base_merge["_FAMILIA_KEY"]
    .isin(
        map_bodega_especial.keys()
    )
)


df_base_merge.loc[
    mask_familia_especial,
    "BODEGA_HECTAREA"
] = (
    df_base_merge.loc[
        mask_familia_especial,
        "_FAMILIA_KEY"
    ]
    .map(map_bodega_especial)
)


# =========================================================
# 13. EXCEPCIÓN GYPSOPHILA URCUQUI
# =========================================================

mask_gypsophila_urcuqui = (
    df_base_merge["_FAMILIA_KEY"]
    .eq("GYPSOPHILA")
    &
    df_base_merge["BODEGA"]
    .eq("URCUQUI")
)


REGISTROS_GYPSO_URCUQUI = int(
    mask_gypsophila_urcuqui.sum()
)


TOTAL_GYPSO_URCUQUI = float(
    df_base_merge.loc[
        mask_gypsophila_urcuqui,
        "_TOTAL_NUM"
    ].sum()
)


df_base_merge.loc[
    mask_gypsophila_urcuqui,
    "BODEGA_HECTAREA"
] = "MALCHINGUI"


# =========================================================
# 14. CREAR LLAVE DE FINCA EN CONSUMOS
# =========================================================

df_base_merge["_BODEGA_HECT_KEY"] = (
    df_base_merge["BODEGA_HECTAREA"]
    .map(normalizar_llave_merge)
)


# =========================================================
# 15. AUDITAR HYPERICUM Y VERONICAS ANTES DEL MERGE
# =========================================================
#
# Solo se exige una llave si existen consumos de esa familia
# en la semana procesada.
#
# Si Hypericum tiene consumos, debe existir:
#
# 2026 | 31 | PYGAN | HYPERICUM
#
# Si Verónicas tiene consumos, debe existir:
#
# 2026 | 31 | PYGAN | VERONICAS
# =========================================================

familias_control_obligatorio = {
    "HYPERICUM",
    "VERONICAS"
}


mask_semana_control = (
    (df_base_merge["_AÑO_CRUCE"] == ANIO_PROCESO)
    &
    (
        df_base_merge["_SEMANA_CRUCE"]
        == SEMANA_PROCESO
    )
)


consumos_control_premerge = (
    df_base_merge.loc[
        mask_semana_control
        &
        df_base_merge["_FAMILIA_KEY"]
        .isin(familias_control_obligatorio)
    ]
    .groupby(
        [
            "_AÑO_CRUCE",
            "_SEMANA_CRUCE",
            "_BODEGA_HECT_KEY",
            "_FAMILIA_KEY"
        ],
        dropna=False
    )
    .agg(
        REGISTROS_CONSUMO=("TOTAL", "size"),
        TOTAL_CONSUMO=("_TOTAL_NUM", "sum")
    )
    .reset_index()
)


hectareas_control_premerge = (
    hectareas_merge.loc[
        (
            hectareas_merge["_AÑO_CRUCE"]
            == ANIO_PROCESO
        )
        &
        (
            hectareas_merge["_SEMANA_CRUCE"]
            == SEMANA_PROCESO
        )
        &
        hectareas_merge["_FAMILIA_KEY"]
        .isin(familias_control_obligatorio)
    ]
    .copy()
)


validacion_familias_premerge = (
    consumos_control_premerge
    .merge(
        hectareas_control_premerge,
        on=llaves_merge,
        how="left",
        indicator="ESTADO_MATCH_PREVIO"
    )
)


validacion_familias_premerge[
    "ESTADO_HECTAREA"
] = np.select(
    [
        validacion_familias_premerge[
            "ESTADO_MATCH_PREVIO"
        ].eq("left_only"),

        validacion_familias_premerge[
            "HECTARIAS"
        ].fillna(0).le(0)
    ],
    [
        "NO EXISTE LA LLAVE",
        "HECTAREA VACIA O CERO"
    ],
    default="OK"
)


print("\n" + "=" * 75)
print(
    f"VALIDACIÓN PREVIA — HYPERICUM Y VERONICAS "
    f"SEMANA {SEMANA_PROCESO}"
)
print("=" * 75)

display(
    validacion_familias_premerge[
        [
            "_AÑO_CRUCE",
            "_SEMANA_CRUCE",
            "_BODEGA_HECT_KEY",
            "_FAMILIA_KEY",
            "REGISTROS_CONSUMO",
            "TOTAL_CONSUMO",
            "HECTARIAS",
            "ESTADO_HECTAREA"
        ]
    ]
)


familias_control_con_error = (
    validacion_familias_premerge.loc[
        ~validacion_familias_premerge[
            "ESTADO_HECTAREA"
        ].eq("OK")
    ]
    .copy()
)


if not familias_control_con_error.empty:

    print("\n⚠️ LLAVES QUE DEBES CORREGIR EN HECTÁREAS:")

    display(
        familias_control_con_error[
            [
                "_AÑO_CRUCE",
                "_SEMANA_CRUCE",
                "_BODEGA_HECT_KEY",
                "_FAMILIA_KEY",
                "REGISTROS_CONSUMO",
                "TOTAL_CONSUMO",
                "HECTARIAS",
                "ESTADO_HECTAREA"
            ]
        ]
    )

    raise ValueError(
        "HYPERICUM o VERONICAS tiene consumos, "
        "pero no dispone de una llave válida con "
        "hectáreas positivas en la base de hectáreas."
    )


print(
    "✅ Las llaves necesarias de HYPERICUM y "
    "VERONICAS existen."
)


# =========================================================
# 16. AUDITAR REASIGNACIONES DE LA SEMANA
# =========================================================

resumen_reasignaciones_semana = (
    df_base_merge.loc[
        mask_semana_control
        &
        (
            df_base_merge["BODEGA_HECTAREA"]
            != df_base_merge["BODEGA"]
        )
    ]
    .groupby(
        [
            "FAMILIA",
            "BODEGA",
            "BODEGA_HECTAREA"
        ],
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("_TOTAL_NUM", "sum")
    )
    .reset_index()
    .sort_values(
        "TOTAL",
        ascending=False
    )
)


print("\n" + "=" * 75)
print(
    f"REASIGNACIONES — SEMANA "
    f"{SEMANA_PROCESO}"
)
print("=" * 75)

display(resumen_reasignaciones_semana)


# =========================================================
# 17. EJECUTAR MERGE PROTEGIDO
# =========================================================

df_cruzado = (
    df_base_merge
    .merge(
        hectareas_merge,
        on=llaves_merge,
        how="left",
        validate="many_to_one",
        indicator="_MATCH_HECTAREA"
    )
)


FILAS_DESPUES_MERGE = len(df_cruzado)


TOTAL_DESPUES_MERGE = float(
    pd.to_numeric(
        df_cruzado["TOTAL"],
        errors="coerce"
    ).sum()
)


# =========================================================
# 18. VALIDAR INTEGRIDAD GLOBAL
# =========================================================

print("\n" + "=" * 75)
print("CONTROL DESPUÉS DEL MERGE")
print("=" * 75)

print(
    f"Filas antes: "
    f"{FILAS_ANTES_MERGE:,}"
)

print(
    f"Filas después: "
    f"{FILAS_DESPUES_MERGE:,}"
)

print(
    f"Total antes: "
    f"${TOTAL_ANTES_MERGE:,.2f}"
)

print(
    f"Total después: "
    f"${TOTAL_DESPUES_MERGE:,.2f}"
)


if FILAS_DESPUES_MERGE != FILAS_ANTES_MERGE:

    raise ValueError(
        "El merge cambió la cantidad de filas."
    )


if not np.isclose(
    TOTAL_ANTES_MERGE,
    TOTAL_DESPUES_MERGE,
    atol=0.01
):

    raise ValueError(
        "El merge cambió el total."
    )


print("✅ El merge no multiplicó registros.")
print("✅ El total permanece igual.")


# =========================================================
# 19. CALCULAR COSTO POR HECTÁREA
# =========================================================

df_cruzado["COSTO_POR_HECTARIA"] = np.where(
    df_cruzado["HECTARIAS"]
    .fillna(0)
    .gt(0),

    df_cruzado["_TOTAL_NUM"]
    /
    df_cruzado["HECTARIAS"],

    np.nan
)


# =========================================================
# 20. AUDITAR LA SEMANA PROCESADA
# =========================================================

df_semana_cruzada = (
    df_cruzado.loc[
        (
            df_cruzado["_AÑO_CRUCE"]
            == ANIO_PROCESO
        )
        &
        (
            df_cruzado["_SEMANA_CRUCE"]
            == SEMANA_PROCESO
        )
    ]
    .copy()
)


FILAS_SEMANA_DESPUES = len(
    df_semana_cruzada
)


TOTAL_SEMANA_DESPUES = float(
    df_semana_cruzada["_TOTAL_NUM"].sum()
)


print("\n" + "=" * 75)
print(
    f"AUDITORÍA SEMANA "
    f"{SEMANA_PROCESO}"
)
print("=" * 75)

print(
    f"Registros antes: "
    f"{REGISTROS_SEMANA_PROCESO:,}"
)

print(
    f"Registros después: "
    f"{FILAS_SEMANA_DESPUES:,}"
)

print(
    f"Total antes: "
    f"${TOTAL_SEMANA_PROCESO:,.2f}"
)

print(
    f"Total después: "
    f"${TOTAL_SEMANA_DESPUES:,.2f}"
)


if FILAS_SEMANA_DESPUES != REGISTROS_SEMANA_PROCESO:

    raise ValueError(
        "Cambió la cantidad de registros de la semana."
    )


if not np.isclose(
    TOTAL_SEMANA_DESPUES,
    TOTAL_SEMANA_PROCESO,
    atol=0.01
):

    raise ValueError(
        "Cambió el total de la semana."
    )


print("✅ La semana continúa cuadrando.")


# =========================================================
# 21. RESULTADO DEL MATCH
# =========================================================

resumen_match_semana = (
    df_semana_cruzada
    .groupby(
        "_MATCH_HECTAREA",
        observed=False,
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("_TOTAL_NUM", "sum")
    )
    .reset_index()
)


print("\n" + "=" * 75)
print("MATCH DE HECTÁREAS — SEMANA PROCESADA")
print("=" * 75)

display(resumen_match_semana)


# =========================================================
# 22. VALIDAR CASOS SIN MATCH
# =========================================================

sin_match_semana = (
    df_semana_cruzada.loc[
        df_semana_cruzada["_MATCH_HECTAREA"]
        .astype("string")
        .eq("left_only")
    ]
    .copy()
)


sin_match_otros = (
    sin_match_semana.loc[
        sin_match_semana["FAMILIA"]
        .eq("OTROS")
    ]
    .copy()
)


sin_match_no_otros = (
    sin_match_semana.loc[
        ~sin_match_semana["FAMILIA"]
        .eq("OTROS")
    ]
    .copy()
)


print("\n" + "=" * 75)
print("REGISTROS SIN MATCH")
print("=" * 75)

print(
    f"Sin match de OTROS: "
    f"{len(sin_match_otros):,}"
)

print(
    f"Sin match de familias distintas de OTROS: "
    f"{len(sin_match_no_otros):,}"
)


if not sin_match_no_otros.empty:

    resumen_sin_match = (
        sin_match_no_otros
        .groupby(
            [
                "BODEGA",
                "BODEGA_HECTAREA",
                "FAMILIA"
            ],
            dropna=False
        )
        .agg(
            REGISTROS=("TOTAL", "size"),
            TOTAL=("_TOTAL_NUM", "sum")
        )
        .reset_index()
        .sort_values(
            "TOTAL",
            ascending=False
        )
    )

    display(resumen_sin_match)

    raise ValueError(
        "Todavía existen familias distintas de OTROS "
        "sin hectáreas."
    )


print(
    "✅ Todos los consumos distintos de OTROS "
    "encontraron su hectárea."
)


# =========================================================
# 23. CONTROL FINAL HYPERICUM Y VERONICAS
# =========================================================

control_hypericum_veronicas = (
    df_semana_cruzada.loc[
        df_semana_cruzada["_FAMILIA_KEY"]
        .isin([
            "HYPERICUM",
            "VERONICAS"
        ])
    ]
    .groupby(
        [
            "FAMILIA",
            "BODEGA",
            "BODEGA_HECTAREA",
            "_MATCH_HECTAREA"
        ],
        observed=True,
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("_TOTAL_NUM", "sum"),
        HECTARIAS=("HECTARIAS", "max")
    )
    .reset_index()
)


# Protección adicional:
# eliminar cualquier combinación vacía generada por Pandas.
control_hypericum_veronicas = (
    control_hypericum_veronicas.loc[
        control_hypericum_veronicas[
            "REGISTROS"
        ].gt(0)
    ]
    .copy()
)


print("\n" + "=" * 75)
print("CONTROL FINAL HYPERICUM Y VERONICAS")
print("=" * 75)

display(control_hypericum_veronicas)


if not control_hypericum_veronicas.empty:

    mask_control_incorrecto = (
        ~control_hypericum_veronicas[
            "BODEGA_HECTAREA"
        ].eq("PYGAN")
        |
        ~control_hypericum_veronicas[
            "_MATCH_HECTAREA"
        ]
        .astype("string")
        .eq("both")
        |
        control_hypericum_veronicas[
            "HECTARIAS"
        ]
        .fillna(0)
        .le(0)
    )

    if mask_control_incorrecto.any():

        print(
            "\n⚠️ REGISTROS REALES CON PROBLEMAS:"
        )

        display(
            control_hypericum_veronicas.loc[
                mask_control_incorrecto
            ]
        )

        raise ValueError(
            "HYPERICUM o VERONICAS no quedó "
            "correctamente cruzada con PYGAN."
        )


print("✅ HYPERICUM quedó asignada a PYGAN.")
print("✅ VERONICAS quedó asignada a PYGAN.")
print("✅ Ambas familias encontraron hectáreas positivas.")


# =========================================================
# 24. CONTROL GYPSOPHILA URCUQUI
# =========================================================

control_gypsophila_urcuqui = (
    df_semana_cruzada.loc[
        df_semana_cruzada["_FAMILIA_KEY"]
        .eq("GYPSOPHILA")
        &
        df_semana_cruzada["BODEGA"]
        .eq("URCUQUI")
    ]
    .groupby(
        [
            "BODEGA",
            "BODEGA_HECTAREA",
            "_MATCH_HECTAREA"
        ],
        observed=False,
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("_TOTAL_NUM", "sum"),
        HECTARIAS=("HECTARIAS", "max")
    )
    .reset_index()
)


print("\n" + "=" * 75)
print("CONTROL GYPSOPHILA URCUQUI")
print("=" * 75)

display(control_gypsophila_urcuqui)


# =========================================================
# 25. ACTUALIZAR DF
# =========================================================

df = df_cruzado.copy()


# =========================================================
# 26. RESULTADO FINAL
# =========================================================

print("\n" + "=" * 75)
print("RESULTADO DEL BLOQUE 6")
print("=" * 75)

print(
    f"Filas finales: "
    f"{len(df):,}"
)

print(
    f"Total final: "
    f"${df['_TOTAL_NUM'].sum():,.2f}"
)

print(
    f"Registros semana "
    f"{SEMANA_PROCESO}: "
    f"{len(df_semana_cruzada):,}"
)

print(
    f"Total semana "
    f"{SEMANA_PROCESO}: "
    f"${TOTAL_SEMANA_DESPUES:,.2f}"
)

print(
    f"Copias redundantes de hectáreas eliminadas: "
    f"{FILAS_REDUNDANTES_ELIMINADAS:,}"
)

print(
    f"Sin match distintos de OTROS: "
    f"{len(sin_match_no_otros):,}"
)

print("✅ VERONICA fue homologada a VERONICAS.")
print("✅ VERONICAS fue asignada a PYGAN.")
print("✅ HYPERICUM fue asignada a PYGAN.")
print("✅ GYPSOPHILA de URCUQUI fue asignada a MALCHINGUI.")
print("✅ No se multiplicaron los consumos.")
print("✅ No cambió el total.")
print("✅ Se calculó COSTO_POR_HECTARIA.")
print("✅ Base actualizada: df.")
print("✅ BLOQUE 6 TERMINADO.")


RECONSTRUCCIÓN DEL PERIODO PARA EL MERGE
Registros cuyo periodo fue reconstruido/corregido: 0
Registros con periodo inválido después de reconstruir: 0
✅ Año y semana de cruce fueron reconstruidos desde FECHA.
✅ Ningún registro queda con periodo inválido para el merge.
CONTROL ANTES DEL MERGE
Filas antes: 376,489
Total antes: $11,145,469.18

VALIDACIÓN DE VALORES DE HECTÁREAS
Valores positivos: 3,652
Valores en cero: 62
Valores vacíos: 4
Textos no numéricos: 0
Valores negativos: 0
✅ Los valores de hectáreas son válidos.

VALIDACIÓN DE LLAVES DE HECTÁREAS
Registros con llave inválida: 0
✅ Todas las llaves de hectáreas son válidas.

DUPLICADOS CON HECTÁREAS DIFERENTES
Llaves con más de un valor numérico: 8
⚠️ Se encontraron valores distintos para una misma llave. El proceso continuará usando la HECTÁREA MÁS ALTA.


,AÑO,SEMANA,FINCA,FAMILIA,HECTARIAS,_BODEGA_HECT_KEY,_FAMILIA_KEY,VALORES_NUMERICOS_DISTINTOS
15,2025,12,URCUQUI,SUNFLOWER,10.86,URCUQUI,SUNFLOWER,2
14,2025,12,URCUQUI,SUNFLOWER,8.023742,URCUQUI,SUNFLOWER,2
0,2025,36,URCUQUI,GYPSOPHILA,4.987664,URCUQUI,GYPSOPHILA,2
1,2025,36,URCUQUI,GYPSOPHILA,1.784023,URCUQUI,GYPSOPHILA,2
2,2025,40,URCUQUI,GYPSOPHILA,2.405814,URCUQUI,GYPSOPHILA,2
3,2025,40,URCUQUI,GYPSOPHILA,0.645462,URCUQUI,GYPSOPHILA,2
4,2025,41,URCUQUI,GYPSOPHILA,2.405814,URCUQUI,GYPSOPHILA,2
5,2025,41,URCUQUI,GYPSOPHILA,0.645462,URCUQUI,GYPSOPHILA,2
6,2025,42,URCUQUI,GYPSOPHILA,2.405814,URCUQUI,GYPSOPHILA,2
7,2025,42,URCUQUI,GYPSOPHILA,0.645462,URCUQUI,GYPSOPHILA,2



CONSOLIDACIÓN DE HECTÁREAS
Filas involucradas en llaves duplicadas: 28
Filas antes: 3,718
Filas después: 3,704
Filas consolidadas: 14

VALOR DEFINITIVO ELEGIDO POR LLAVE:


,_AÑO_CRUCE,_SEMANA_CRUCE,_BODEGA_HECT_KEY,_FAMILIA_KEY,VALORES_NUMERICOS_DISTINTOS,HECTARIA_ELEGIDA_MAX
0,2025,12,URCUQUI,SUNFLOWER,2,10.86
1,2025,36,URCUQUI,GYPSOPHILA,2,4.987664
2,2025,40,URCUQUI,GYPSOPHILA,2,2.405814
3,2025,41,URCUQUI,GYPSOPHILA,2,2.405814
4,2025,42,URCUQUI,GYPSOPHILA,2,2.405814
5,2025,43,URCUQUI,GYPSOPHILA,2,2.405814
6,2025,44,URCUQUI,GYPSOPHILA,2,2.901914
7,2025,45,URCUQUI,GYPSOPHILA,2,3.347089


✅ hectareas_merge quedó única por llave.
✅ Cuando existían varios valores, se conservó la hectárea más alta.

VALIDACIÓN PREVIA — HYPERICUM Y VERONICAS SEMANA 34


...


REGISTROS SIN MATCH
Sin match de OTROS: 1,604
Sin match de familias distintas de OTROS: 0
✅ Todos los consumos distintos de OTROS encontraron su hectárea.

CONTROL FINAL HYPERICUM Y VERONICAS


,FAMILIA,BODEGA,BODEGA_HECTAREA,_MATCH_HECTAREA,REGISTROS,TOTAL,HECTARIAS
0,HYPERICUM,MALCHINGUI,PYGAN,both,128,1251.39,5.75
1,HYPERICUM,PYGAN,PYGAN,both,112,3772.67,5.75
2,VERONICAS,MALCHINGUI,PYGAN,both,165,1837.61,4.83
3,VERONICAS,PYGAN,PYGAN,both,137,3363.75,4.83


✅ HYPERICUM quedó asignada a PYGAN.
✅ VERONICAS quedó asignada a PYGAN.
✅ Ambas familias encontraron hectáreas positivas.

CONTROL GYPSOPHILA URCUQUI


,BODEGA,BODEGA_HECTAREA,_MATCH_HECTAREA,REGISTROS,TOTAL,HECTARIAS



RESULTADO DEL BLOQUE 6
Filas finales: 376,489
Total final: $11,145,472.00
Registros semana 34: 12,042
Total semana 34: $336,487.66
Copias redundantes de hectáreas eliminadas: 14
Sin match distintos de OTROS: 0
✅ VERONICA fue homologada a VERONICAS.
✅ VERONICAS fue asignada a PYGAN.
✅ HYPERICUM fue asignada a PYGAN.
✅ GYPSOPHILA de URCUQUI fue asignada a MALCHINGUI.
✅ No se multiplicaron los consumos.
✅ No cambió el total.
✅ Se calculó COSTO_POR_HECTARIA.
✅ Base actualizada: df.
✅ BLOQUE 6 TERMINADO.


# BLOQUE 7

In [ ]:
# BLOQUE 7 — EXPORTACIÓN DEFINITIVA PARA POWER BI
# =========================================================
#
# CONTRATO CORRECTO:
#
# - Hoja: Sheet1
# - 20 columnas
# - Dos encabezados llamados FINCA
#
# Primera FINCA:
#     BODEGA_HECTAREA
#     La utiliza el dashboard.
#
# Segunda FINCA:
#     BODEGA
#     Power BI la reconocerá como FINCA_1.
#
# NO se debe usar "Obtener datos" para cargar este archivo.
# El archivo debe reemplazar físicamente la fuente anterior.
# =========================================================


# =========================================================
# 1. IMPORTACIONES
# =========================================================

import os
import pandas as pd
import numpy as np

from openpyxl import load_workbook
from google.colab import files


# =========================================================
# FUNCIONES ROBUSTAS PARA LA FUENTE ACTUAL
# =========================================================

def convertir_numero_mixto_b7(serie):
    """Convierte números normales y textos con coma decimal."""
    salida = pd.to_numeric(serie, errors="coerce")

    texto = (
        serie.astype("string")
        .str.strip()
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
    )

    mask_contenido = serie.notna() & texto.notna() & texto.ne("")
    mask_pendiente = mask_contenido & salida.isna()

    if mask_pendiente.any():
        pendiente = texto.loc[mask_pendiente].copy()

        tiene_coma = pendiente.str.contains(",", regex=False, na=False)
        tiene_punto = pendiente.str.contains(".", regex=False, na=False)

        mask_solo_coma = tiene_coma & ~tiene_punto
        pendiente.loc[mask_solo_coma] = (
            pendiente.loc[mask_solo_coma]
            .str.replace(",", ".", regex=False)
        )

        mask_ambos = tiene_coma & tiene_punto
        if mask_ambos.any():
            ambos = pendiente.loc[mask_ambos].copy()
            coma_ultima = ambos.str.rfind(",") > ambos.str.rfind(".")

            idx_coma_decimal = ambos.index[coma_ultima]
            idx_punto_decimal = ambos.index[~coma_ultima]

            pendiente.loc[idx_coma_decimal] = (
                pendiente.loc[idx_coma_decimal]
                .str.replace(".", "", regex=False)
                .str.replace(",", ".", regex=False)
            )

            pendiente.loc[idx_punto_decimal] = (
                pendiente.loc[idx_punto_decimal]
                .str.replace(",", "", regex=False)
            )

        salida.loc[mask_pendiente] = pd.to_numeric(
            pendiente,
            errors="coerce"
        )

    return salida


def convertir_fecha_mixta_b7(serie):
    """Convierte seriales Excel y fechas escritas."""
    resultado = pd.Series(
        pd.NaT,
        index=serie.index,
        dtype="datetime64[ns]"
    )

    numerica = pd.to_numeric(serie, errors="coerce")
    mask_serial_excel = numerica.notna()

    if mask_serial_excel.any():
        resultado.loc[mask_serial_excel] = pd.to_datetime(
            numerica.loc[mask_serial_excel],
            unit="D",
            origin="1899-12-30",
            errors="coerce"
        )

    mask_texto = (
        ~mask_serial_excel
        & serie.notna()
        & serie.astype("string").str.strip().ne("")
    )

    if mask_texto.any():
        resultado.loc[mask_texto] = pd.to_datetime(
            serie.loc[mask_texto].astype("string").str.strip(),
            errors="coerce"
        )

    return resultado


# =========================================================
# 2. VALIDAR VARIABLES DEL BLOQUE 6
# =========================================================

variables_necesarias_b7 = [
    "df",
    "ANIO_PROCESO",
    "SEMANA_PROCESO",
    "REGISTROS_SEMANA_PROCESO",
    "TOTAL_SEMANA_PROCESO"
]

variables_faltantes_b7 = [
    variable
    for variable in variables_necesarias_b7
    if variable not in globals()
]

if variables_faltantes_b7:

    raise ValueError(
        f"Faltan variables de los bloques anteriores: "
        f"{variables_faltantes_b7}"
    )


columnas_necesarias_df = [
    "FECHA",
    "BODEGA",
    "BODEGA_HECTAREA",
    "GRUPOS DE MATERIALES",
    "NOMBRE",
    "TOTAL",
    "CENTRO DE COSTO",
    "FAMILIA",
    "FAMILIA_DESGLOSE",
    "HECTARIAS",
    "COSTO_POR_HECTARIA",
    "MES",
    "DOC",
    "CODIGO",
    "UND",
    "CANTIDAD",
    "PROM",
    "USUARIO",
    "AREA",
    "_AÑO_CRUCE",
    "_SEMANA_CRUCE",
    "_TOTAL_NUM"
]

columnas_faltantes_df = [
    columna
    for columna in columnas_necesarias_df
    if columna not in df.columns
]

if columnas_faltantes_df:

    raise ValueError(
        f"Faltan columnas necesarias en df: "
        f"{columnas_faltantes_df}"
    )

print("✅ Se encontraron todas las columnas necesarias.")


# =========================================================
# 3. CONTROL ANTES DE EXPORTAR
# =========================================================

FILAS_ANTES_EXPORTAR = len(df)

TOTAL_ANTES_EXPORTAR = float(
    pd.to_numeric(
        df["_TOTAL_NUM"],
        errors="coerce"
    ).sum()
)

print("\n" + "=" * 75)
print("CONTROL ANTES DE CONSTRUIR LA FUENTE POWER BI")
print("=" * 75)

print(
    f"Filas actuales: "
    f"{FILAS_ANTES_EXPORTAR:,}"
)

print(
    f"Total actual: "
    f"${TOTAL_ANTES_EXPORTAR:,.2f}"
)


# =========================================================
# 4. VALIDAR AÑO Y SEMANA
# =========================================================

mask_periodo_invalido = (
    df["_AÑO_CRUCE"].isna()
    |
    df["_SEMANA_CRUCE"].isna()
    |
    ~df["_SEMANA_CRUCE"].between(1, 53)
)

cantidad_periodos_invalidos = int(
    mask_periodo_invalido.sum()
)

print("\n" + "=" * 75)
print("VALIDACIÓN DE AÑO Y SEMANA")
print("=" * 75)

print(
    f"Registros con periodo inválido: "
    f"{cantidad_periodos_invalidos:,}"
)

if cantidad_periodos_invalidos > 0:

    display(
        df.loc[
            mask_periodo_invalido,
            [
                "FECHA",
                "SEMANA",
                "_AÑO_CRUCE",
                "_SEMANA_CRUCE",
                "TOTAL"
            ]
        ].head(100)
    )

    raise ValueError(
        "Existen registros con año o semana inválida."
    )

print("✅ Todos los registros tienen año y semana válidos.")


# =========================================================
# 5. CONSTRUIR ANIOSEMANA COMO TEXTO
# =========================================================
#
# Ejemplos:
# 2026 + semana 2  = 202602
# 2026 + semana 31 = 202631
# =========================================================

aniosemana_powerbi = (
    df["_AÑO_CRUCE"]
    .astype("Int64")
    .astype("string")
    +
    df["_SEMANA_CRUCE"]
    .astype("Int64")
    .astype("string")
    .str.zfill(2)
)

mask_aniosemana_invalido = (
    aniosemana_powerbi.isna()
    |
    ~aniosemana_powerbi.str.match(
        r"^\d{6}$",
        na=False
    )
)

if mask_aniosemana_invalido.any():

    display(
        df.loc[
            mask_aniosemana_invalido,
            [
                "FECHA",
                "_AÑO_CRUCE",
                "_SEMANA_CRUCE"
            ]
        ].head(100)
    )

    raise ValueError(
        "No fue posible construir correctamente ANIOSEMANA."
    )

print("✅ ANIOSEMANA fue construido correctamente.")


# =========================================================
# 6. CONTROL HYPERICUM Y VERONICAS ANTES DE EXPORTAR
# =========================================================

mask_semana_control = (
    (df["_AÑO_CRUCE"] == ANIO_PROCESO)
    &
    (df["_SEMANA_CRUCE"] == SEMANA_PROCESO)
)

control_hypericum_veronicas = (
    df.loc[
        mask_semana_control
        &
        df["FAMILIA"].isin(
            [
                "HYPERICUM",
                "VERONICAS"
            ]
        ),
        [
            "FAMILIA",
            "BODEGA",
            "BODEGA_HECTAREA",
            "HECTARIAS",
            "TOTAL"
        ]
    ]
    .copy()
)

control_hypericum_veronicas["TOTAL"] = convertir_numero_mixto_b7(
    control_hypericum_veronicas["TOTAL"]
)

resumen_hypericum_veronicas = (
    control_hypericum_veronicas
    .groupby(
        [
            "FAMILIA",
            "BODEGA",
            "BODEGA_HECTAREA"
        ],
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("TOTAL", "sum"),
        HECTARIAS=("HECTARIAS", "max")
    )
    .reset_index()
    .sort_values(
        [
            "FAMILIA",
            "BODEGA",
            "BODEGA_HECTAREA"
        ]
    )
)

print("\n" + "=" * 75)
print("CONTROL HYPERICUM Y VERONICAS")
print("=" * 75)

display(resumen_hypericum_veronicas)

if not control_hypericum_veronicas.empty:

    mask_control_incorrecto = (
        ~control_hypericum_veronicas[
            "BODEGA_HECTAREA"
        ].eq("PYGAN")
        |
        pd.to_numeric(
            control_hypericum_veronicas["HECTARIAS"],
            errors="coerce"
        )
        .fillna(0)
        .le(0)
    )

    if mask_control_incorrecto.any():

        print("\n⚠️ REGISTROS CON PROBLEMAS:")

        display(
            control_hypericum_veronicas.loc[
                mask_control_incorrecto
            ]
        )

        raise ValueError(
            "HYPERICUM o VERONICAS no está correctamente "
            "asignada a PYGAN con hectáreas positivas."
        )

print("✅ HYPERICUM será exportado en PYGAN.")
print("✅ VERONICAS será exportado en PYGAN.")


# =========================================================
# 7. CONSTRUIR BASE POWER BI
# =========================================================
#
# MUY IMPORTANTE:
#
# Primera FINCA  = BODEGA_HECTAREA
# Segunda FINCA  = BODEGA
#
# En Excel ambas se llaman FINCA.
# Power BI reconocerá la segunda como FINCA_1.
# =========================================================

base_powerbi = pd.concat(
    [
        # 1
        df["FECHA"],

        # 2
        aniosemana_powerbi,

        # 3 — FINCA PRINCIPAL DEL DASHBOARD
        df["BODEGA_HECTAREA"],

        # 4 — FINCA CONTABLE / FINCA_1 EN POWER BI
        df["BODEGA"],

        # 5
        df["GRUPOS DE MATERIALES"],

        # 6
        df["NOMBRE"],

        # 7
        df["TOTAL"],

        # 8
        df["CENTRO DE COSTO"],

        # 9
        df["FAMILIA"],

        # 10
        df["FAMILIA_DESGLOSE"],

        # 11
        df["HECTARIAS"],

        # 12
        df["COSTO_POR_HECTARIA"],

        # 13
        df["MES"],

        # 14
        df["DOC"],

        # 15
        df["CODIGO"],

        # 16
        df["UND"],

        # 17
        df["CANTIDAD"],

        # 18
        df["PROM"],

        # 19
        df["USUARIO"],

        # 20
        df["AREA"]
    ],
    axis=1
)


# =========================================================
# 8. ASIGNAR HEADERS EXACTOS
# =========================================================

HEADERS_POWER_BI = [
    "FECHA",
    "ANIOSEMANA",
    "FINCA",
    "FINCA",
    "GRUPOS DE MATERIALES",
    "NOMBRE",
    "TOTAL",
    "CENTRO DE COSTO",
    "FAMILIA",
    "FAMILIA_DESGLOSE",
    "HECTARIAS",
    "COSTO_POR_HECTARIA",
    "MES",
    "DOC",
    "CODIGO",
    "UND",
    "CANTIDAD",
    "PROM",
    "USUARIO",
    "AREA"
]

base_powerbi.columns = HEADERS_POWER_BI


# =========================================================
# 9. VALIDAR ESTRUCTURA
# =========================================================

print("\n" + "=" * 75)
print("AUDITORÍA DE HEADERS POWER BI")
print("=" * 75)

print(
    f"Cantidad de columnas: "
    f"{len(base_powerbi.columns)}"
)

for numero, header in enumerate(
    base_powerbi.columns,
    start=1
):
    print(f"{numero:>2}. {header}")

if len(base_powerbi.columns) != 20:

    raise ValueError(
        "La fuente debe contener exactamente 20 columnas."
    )

if list(base_powerbi.columns) != HEADERS_POWER_BI:

    raise ValueError(
        "Los headers o el orden no coinciden con "
        "la estructura esperada por Power BI."
    )

cantidad_fincas = list(
    base_powerbi.columns
).count("FINCA")

if cantidad_fincas != 2:

    raise ValueError(
        "La fuente debe tener exactamente dos "
        "encabezados llamados FINCA."
    )

print("✅ La base tiene exactamente 20 columnas.")
print("✅ Existen exactamente dos headers FINCA.")
print("✅ Power BI podrá crear FINCA y FINCA_1.")


# =========================================================
# 10. VALIDAR CONTENIDO DE LAS DOS FINCAS
# =========================================================

primera_finca = (
    base_powerbi.iloc[:, 2]
    .astype("string")
    .str.strip()
)

segunda_finca = (
    base_powerbi.iloc[:, 3]
    .astype("string")
    .str.strip()
)

mask_primera_finca_vacia = (
    primera_finca.isna()
    |
    primera_finca.eq("")
)

mask_segunda_finca_vacia = (
    segunda_finca.isna()
    |
    segunda_finca.eq("")
)

if mask_primera_finca_vacia.any():

    display(
        base_powerbi.loc[
            mask_primera_finca_vacia
        ].head(100)
    )

    raise ValueError(
        "Existen registros sin la primera FINCA."
    )

if mask_segunda_finca_vacia.any():

    display(
        base_powerbi.loc[
            mask_segunda_finca_vacia
        ].head(100)
    )

    raise ValueError(
        "Existen registros sin la segunda FINCA."
    )


fincas_permitidas = {
    "MALCHINGUI",
    "PYGAN",
    "URCUQUI"
}

fincas_primera_columna = set(
    primera_finca
    .dropna()
    .unique()
    .tolist()
)

fincas_segunda_columna = set(
    segunda_finca
    .dropna()
    .unique()
    .tolist()
)

fincas_no_permitidas = (
    fincas_primera_columna
    .union(fincas_segunda_columna)
    - fincas_permitidas
)

if fincas_no_permitidas:

    raise ValueError(
        f"Existen fincas no reconocidas: "
        f"{sorted(fincas_no_permitidas)}"
    )

print(
    "✅ Primera FINCA contiene la finca productiva."
)

print(
    "✅ Segunda FINCA contiene la finca contable."
)


# =========================================================
# 11. GARANTIZAR TIPOS DE DATOS
# =========================================================

fecha_powerbi = convertir_fecha_mixta_b7(
    base_powerbi.iloc[:, 0]
)

if fecha_powerbi.isna().any():

    display(
        base_powerbi.loc[
            fecha_powerbi.isna()
        ].head(100)
    )

    raise ValueError(
        "Existen fechas inválidas en la base final."
    )

base_powerbi.iloc[:, 0] = fecha_powerbi


base_powerbi.iloc[:, 1] = (
    base_powerbi.iloc[:, 1]
    .astype("string")
)


posiciones_numericas = [
    6,   # TOTAL
    10,  # HECTARIAS
    11,  # COSTO_POR_HECTARIA
    13,  # DOC
    14,  # CODIGO
    16,  # CANTIDAD
    17   # PROM
]


for posicion in posiciones_numericas:

    columna_original = (
        base_powerbi.iloc[:, posicion]
        .copy()
    )

    columna_numerica = convertir_numero_mixto_b7(
        columna_original
    )

    mask_contenido = (
        columna_original.notna()
        &
        columna_original
        .astype("string")
        .str.strip()
        .ne("")
    )

    mask_conversion_invalida = (
        mask_contenido
        &
        columna_numerica.isna()
    )

    if mask_conversion_invalida.any():

        nombre_columna = HEADERS_POWER_BI[
            posicion
        ]

        print(
            f"\n⚠️ Valores no numéricos en "
            f"{nombre_columna}:"
        )

        display(
            columna_original.loc[
                mask_conversion_invalida
            ].head(100)
        )

        raise ValueError(
            f"No se pudo convertir correctamente "
            f"la columna {nombre_columna}."
        )

    base_powerbi.iloc[:, posicion] = (
        columna_numerica
    )

print("✅ Los tipos de datos fueron validados.")


# =========================================================
# 12. VALIDACIÓN GLOBAL DE FILAS Y TOTAL
# =========================================================

FILAS_BASE_POWERBI = len(
    base_powerbi
)

TOTAL_BASE_POWERBI = float(
    pd.to_numeric(
        base_powerbi.iloc[:, 6],
        errors="coerce"
    ).sum()
)

print("\n" + "=" * 75)
print("CONTROL GLOBAL DE LA FUENTE POWER BI")
print("=" * 75)

print(
    f"Filas antes: "
    f"{FILAS_ANTES_EXPORTAR:,}"
)

print(
    f"Filas finales: "
    f"{FILAS_BASE_POWERBI:,}"
)

print(
    f"Total antes: "
    f"${TOTAL_ANTES_EXPORTAR:,.2f}"
)

print(
    f"Total final: "
    f"${TOTAL_BASE_POWERBI:,.2f}"
)

if FILAS_BASE_POWERBI != FILAS_ANTES_EXPORTAR:

    raise ValueError(
        "La construcción de la fuente cambió "
        "la cantidad de filas."
    )

if not np.isclose(
    TOTAL_BASE_POWERBI,
    TOTAL_ANTES_EXPORTAR,
    atol=0.01
):

    raise ValueError(
        "La construcción de la fuente cambió el total."
    )

print("✅ No se perdió ni duplicó ningún consumo.")


# =========================================================
# 13. VALIDAR SEMANA PROCESADA
# =========================================================

ANIOSEMANA_PROCESO = (
    f"{ANIO_PROCESO}"
    f"{str(SEMANA_PROCESO).zfill(2)}"
)

mask_semana_powerbi = (
    base_powerbi.iloc[:, 1]
    .astype("string")
    .eq(ANIOSEMANA_PROCESO)
)

base_powerbi_semana = (
    base_powerbi.loc[
        mask_semana_powerbi
    ]
    .copy()
)

FILAS_SEMANA_POWERBI = len(
    base_powerbi_semana
)

TOTAL_SEMANA_POWERBI = float(
    pd.to_numeric(
        base_powerbi_semana.iloc[:, 6],
        errors="coerce"
    ).sum()
)

print("\n" + "=" * 75)
print(
    f"CONTROL FINAL — ANIOSEMANA "
    f"{ANIOSEMANA_PROCESO}"
)
print("=" * 75)

print(
    f"Registros esperados: "
    f"{REGISTROS_SEMANA_PROCESO:,}"
)

print(
    f"Registros finales: "
    f"{FILAS_SEMANA_POWERBI:,}"
)

print(
    f"Total esperado: "
    f"${TOTAL_SEMANA_PROCESO:,.2f}"
)

print(
    f"Total final: "
    f"${TOTAL_SEMANA_POWERBI:,.2f}"
)

if FILAS_SEMANA_POWERBI != REGISTROS_SEMANA_PROCESO:

    raise ValueError(
        "La semana procesada no conserva "
        "la cantidad de registros esperada."
    )

if not np.isclose(
    TOTAL_SEMANA_POWERBI,
    TOTAL_SEMANA_PROCESO,
    atol=0.01
):

    raise ValueError(
        "La semana procesada no conserva "
        "el total esperado."
    )

print("✅ La semana procesada continúa cuadrando.")


# =========================================================
# 14. RESUMEN DE LAS DOS COLUMNAS FINCA
# =========================================================

resumen_fincas_powerbi = (
    pd.DataFrame({
        "FINCA_DASHBOARD": (
            base_powerbi_semana.iloc[:, 2]
        ),
        "FINCA_CONTABLE_FINCA_1": (
            base_powerbi_semana.iloc[:, 3]
        ),
        "TOTAL": pd.to_numeric(
            base_powerbi_semana.iloc[:, 6],
            errors="coerce"
        )
    })
    .groupby(
        [
            "FINCA_DASHBOARD",
            "FINCA_CONTABLE_FINCA_1"
        ],
        dropna=False
    )
    .agg(
        REGISTROS=("TOTAL", "size"),
        TOTAL=("TOTAL", "sum")
    )
    .reset_index()
    .sort_values(
        [
            "FINCA_DASHBOARD",
            "FINCA_CONTABLE_FINCA_1"
        ]
    )
)

print("\n" + "=" * 75)
print("CONTROL DE FINCA Y FINCA_1")
print("=" * 75)

display(resumen_fincas_powerbi)


# =========================================================
# 15. DEFINIR ARCHIVO DE SALIDA
# =========================================================

nombre_archivo = (
    "base_costos_hectareas_2026_final.xlsx"
)

archivo_salida = os.path.join(
    "/content",
    nombre_archivo
)

NOMBRE_HOJA_POWERBI = "Sheet1"


if os.path.exists(archivo_salida):

    os.remove(archivo_salida)


print("\n" + "=" * 75)
print("GENERANDO ARCHIVO EXCEL")
print("=" * 75)

print(
    f"Ruta de salida: "
    f"{archivo_salida}"
)


# =========================================================
# 16. EXPORTAR EXCEL
# =========================================================

base_powerbi.to_excel(
    archivo_salida,
    index=False,
    sheet_name=NOMBRE_HOJA_POWERBI,
    freeze_panes=(1, 0),
    engine="openpyxl"
)

if not os.path.exists(archivo_salida):

    raise FileNotFoundError(
        "El archivo Excel no fue creado."
    )

tamanio_archivo_mb = (
    os.path.getsize(archivo_salida)
    / 1024
    / 1024
)

print("✅ El archivo fue escrito correctamente.")

print(
    f"Tamaño del archivo: "
    f"{tamanio_archivo_mb:.2f} MB"
)


# =========================================================
# 17. VERIFICAR ARCHIVO ESCRITO
# =========================================================

libro_verificacion = load_workbook(
    archivo_salida,
    read_only=True,
    data_only=True
)

hojas_escritas = (
    libro_verificacion.sheetnames
)

if hojas_escritas != ["Sheet1"]:

    libro_verificacion.close()

    raise ValueError(
        "El archivo no contiene únicamente Sheet1."
    )


hoja_verificacion = (
    libro_verificacion["Sheet1"]
)

headers_escritos = [
    hoja_verificacion.cell(
        row=1,
        column=columna
    ).value

    for columna in range(
        1,
        hoja_verificacion.max_column + 1
    )
]

filas_escritas = (
    hoja_verificacion.max_row - 1
)

columnas_escritas = (
    hoja_verificacion.max_column
)

libro_verificacion.close()


print("\n" + "=" * 75)
print("VALIDACIÓN DEL ARCHIVO ESCRITO")
print("=" * 75)

print(
    f"Filas escritas: "
    f"{filas_escritas:,}"
)

print(
    f"Columnas escritas: "
    f"{columnas_escritas:,}"
)

print(
    f"Headers escritos: "
    f"{headers_escritos}"
)


if headers_escritos != HEADERS_POWER_BI:

    raise ValueError(
        "Los headers escritos no coinciden exactamente "
        "con los esperados por Power BI."
    )

if filas_escritas != len(base_powerbi):

    raise ValueError(
        "La cantidad de filas escrita es incorrecta."
    )

if columnas_escritas != 20:

    raise ValueError(
        "El archivo debe contener exactamente 20 columnas."
    )

if headers_escritos.count("FINCA") != 2:

    raise ValueError(
        "El archivo debe contener exactamente "
        "dos headers FINCA."
    )

print("✅ La hoja se llama exactamente Sheet1.")
print("✅ El archivo tiene exactamente 20 columnas.")
print("✅ Se escribieron dos encabezados FINCA.")
print("✅ Power BI podrá reconocer FINCA y FINCA_1.")
print("✅ Los headers coinciden exactamente.")


# =========================================================
# 18. RESULTADO FINAL
# =========================================================

print("\n" + "=" * 75)
print("EXPORTACIÓN CORRECTA PARA POWER BI TERMINADA")
print("=" * 75)

print(
    f"Archivo generado: "
    f"{archivo_salida}"
)

print(
    f"Hoja generada: "
    f"{NOMBRE_HOJA_POWERBI}"
)

print(
    f"Filas exportadas: "
    f"{len(base_powerbi):,}"
)

print(
    f"Columnas exportadas: "
    f"{len(base_powerbi.columns):,}"
)

print(
    f"Total exportado: "
    f"${TOTAL_BASE_POWERBI:,.2f}"
)

print(
    f"ANIOSEMANA procesada: "
    f"{ANIOSEMANA_PROCESO}"
)

print(
    f"Registros semana procesada: "
    f"{FILAS_SEMANA_POWERBI:,}"
)

print(
    f"Total semana procesada: "
    f"${TOTAL_SEMANA_POWERBI:,.2f}"
)

print(
    f"Tamaño del archivo: "
    f"{tamanio_archivo_mb:.2f} MB"
)

print("✅ Primera FINCA = BODEGA_HECTAREA.")
print("✅ Segunda FINCA = BODEGA.")
print("✅ Power BI reconocerá la segunda como FINCA_1.")
print("✅ HYPERICUM aparecerá en PYGAN.")
print("✅ VERONICAS aparecerá en PYGAN.")
print("✅ Se conservaron las 20 columnas.")
print("✅ Se conservó la hoja Sheet1.")
print("✅ El archivo puede reemplazar la fuente anterior.")


# =========================================================
# 19. DESCARGAR
# =========================================================

print("\n" + "=" * 75)
print("INICIANDO DESCARGA")
print("=" * 75)

print(
    f"Descargando: "
    f"{nombre_archivo}"
)

files.download(archivo_salida)

✅ Se encontraron todas las columnas necesarias.

CONTROL ANTES DE CONSTRUIR LA FUENTE POWER BI
Filas actuales: 376,489
Total actual: $11,145,472.00

VALIDACIÓN DE AÑO Y SEMANA
Registros con periodo inválido: 0
✅ Todos los registros tienen año y semana válidos.
✅ ANIOSEMANA fue construido correctamente.

CONTROL HYPERICUM Y VERONICAS


,FAMILIA,BODEGA,BODEGA_HECTAREA,REGISTROS,TOTAL,HECTARIAS
0,HYPERICUM,MALCHINGUI,PYGAN,128,1251.39,5.75
1,HYPERICUM,PYGAN,PYGAN,112,3772.67,5.75
2,VERONICAS,MALCHINGUI,PYGAN,165,1837.61,4.83
3,VERONICAS,PYGAN,PYGAN,137,3363.75,4.83


✅ HYPERICUM será exportado en PYGAN.
✅ VERONICAS será exportado en PYGAN.

AUDITORÍA DE HEADERS POWER BI
Cantidad de columnas: 20
 1. FECHA
 2. ANIOSEMANA
 3. FINCA
 4. FINCA
 5. GRUPOS DE MATERIALES
 6. NOMBRE
 7. TOTAL
 8. CENTRO DE COSTO
 9. FAMILIA
10. FAMILIA_DESGLOSE
11. HECTARIAS
12. COSTO_POR_HECTARIA
13. MES
14. DOC
15. CODIGO
16. UND
17. CANTIDAD
18. PROM
19. USUARIO
20. AREA
✅ La base tiene exactamente 20 columnas.
✅ Existen exactamente dos headers FINCA.
✅ Power BI podrá crear FINCA y FINCA_1.
✅ Primera FINCA contiene la finca productiva.
✅ Segunda FINCA contiene la finca contable.
✅ Los tipos de datos fueron validados.

CONTROL GLOBAL DE LA FUENTE POWER BI
Filas antes: 376,489
Filas finales: 376,489
Total antes: $11,145,472.00
Total final: $11,145,472.00
✅ No se perdió ni duplicó ningún consumo.

CONTROL FINAL — ANIOSEMANA 202634
Registros esperados: 12,042
Registros finales: 12,042
Total esperado: $336,487.66
Total final: $336,487.66
✅ La semana procesada continúa cuadran

,FINCA_DASHBOARD,FINCA_CONTABLE_FINCA_1,REGISTROS,TOTAL
0,MALCHINGUI,MALCHINGUI,8063,249802.89
1,MALCHINGUI,URCUQUI,1,9.49
2,PYGAN,MALCHINGUI,293,3089.00
3,PYGAN,PYGAN,714,23127.99
4,URCUQUI,MALCHINGUI,306,11756.75
5,URCUQUI,URCUQUI,2665,48701.54



GENERANDO ARCHIVO EXCEL
Ruta de salida: /content/base_costos_hectareas_2026_final.xlsx
✅ El archivo fue escrito correctamente.
Tamaño del archivo: 33.02 MB

VALIDACIÓN DEL ARCHIVO ESCRITO
Filas escritas: 376,489
Columnas escritas: 20
Headers escritos: ['FECHA', 'ANIOSEMANA', 'FINCA', 'FINCA', 'GRUPOS DE MATERIALES', 'NOMBRE', 'TOTAL', 'CENTRO DE COSTO', 'FAMILIA', 'FAMILIA_DESGLOSE', 'HECTARIAS', 'COSTO_POR_HECTARIA', 'MES', 'DOC', 'CODIGO', 'UND', 'CANTIDAD', 'PROM', 'USUARIO', 'AREA']
✅ La hoja se llama exactamente Sheet1.
✅ El archivo tiene exactamente 20 columnas.
✅ Se escribieron dos encabezados FINCA.
✅ Power BI podrá reconocer FINCA y FINCA_1.
✅ Los headers coinciden exactamente.

EXPORTACIÓN CORRECTA PARA POWER BI TERMINADA
Archivo generado: /content/base_costos_hectareas_2026_final.xlsx
Hoja generada: Sheet1
Filas exportadas: 376,489
Columnas exportadas: 20
Total exportado: $11,145,472.00
ANIOSEMANA procesada: 202634
Registros semana procesada: 12,042
Total semana procesada: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>